# 第8章 接触角と実験計画法・ベイズ最適化（ノートブック版）

教科書の第8章の 8-6 のプログラムを、本文の `run_ch8.sh` の流れのとおりに1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約1分。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 合成の滴の画像で、接触角の計算を確かめる（球冠・Young–Laplace の形・映り込み） | A の 2 | `drop_synth.py`・`contact_angle.py` |
| 2 | 合成の1円玉の影絵で、スケール（mm/画素）を求める | B の 4（合成画像で） | `drop_synth.py --coin`・`contact_angle.py --calib` |
| 3 | 2<sup>3</sup> 要因計画：実験表と、合成データでの分散分析・疑似反復・交互作用 | D の 7・11（`--demo`） | `design_ch8.py`・`doe_anova.py` |
| 4 | 疎水性回復：合成データでの非線形回帰と状態空間モデル | E の 14（`--demo`） | `recovery_fit.py` |
| 5 | 合成の曲面でのベイズ最適化（**ノートブックでは 10 回くり返しの試しの設定**） | F の 15 | `bo_core.py`・`bo_sim.py` |
| 6 | 研磨条件の次の候補を選ぶ（合成の結果で練習） | F の 16・17（`--demo`） | `bo_next.py` |

- 次のセルたちが、8-6 の8つのファイルを、このノートブックの隣に `%%writefile` で書き出します（教科書の `setup_ch8.sh` で `~/home-lab/ch8/` に保存する代わり）。中身は 8-6 と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- 各節では、`run_ch8.sh` の `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（中では `python ○○.py …` を別のプロセスで動かします）。表示を見てから、`ok()` で教科書の手順と 8-7 の「確かめること」を確かめます。図は `check/` と、このノートブックの隣に保存され、セルの下にも表示します。
- **5 節のベイズ最適化のくり返しは重い**ので、ノートブックでは試しの設定（`--reps 10`）を既定にしています。教科書の本番（`--reps 30`、図8-12 の値。執筆時の Mac で約16秒、Colab ではその数倍）にするときは、セルの `REPS = 10` を `REPS = 30` に書きかえてください。くり返しの r 回目は種 `1000 + r` から始まるので、10 回の結果は 30 回の最初の 10 回と同じです。ノートブックの確かめは、くり返しの数に関係なく成り立つことにし、30 回のときだけ教科書の値とも比べます。
- **入れていないもの（Colab ではできない手順）**：撮影台を組んでスマホで水滴を撮ること（手順 B の 3、C の 5・6、D の 8〜10、E の 12・13、F の 16・17 の実物の研磨と撮影）、iPhone の HEIC を `sips` で JPEG にする行、自分の写真を測る `contact_angle.py photos/…` と、その結果の `doe_anova.py design_ch8.csv angles.csv`・`recovery_fit.py angles_rec.csv`・`bo_next.py bo_runs.csv`。ここでは、かわりに合成の滴の画像（`drop_synth.py`）と、各プログラムの `--demo`（合成データ）で同じ解析を動かします。自分の写真を測るときは、Mac で教科書の手順のとおりに行ってください（写真を Colab の左のファイルの欄に上げて `run("contact_angle.py 写真.jpg --csv angles.csv --mm-per-px …")` としても動くはずですが、執筆時には確かめていません）。
- ⚠ 実物の実験の安全の注意（エタノール・IPA の火気、ガラスのふち、紙やすりの粉など）は、教科書の 8-4 の危険ボックスを読んでください。このノートブックは計算だけです。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、OpenCV・scipy・scikit-learn・statsmodels の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・scikit-learn・statsmodels・OpenCV が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("cv2", "opencv-python-headless"), ("sklearn", "scikit-learn"), ("statsmodels", "statsmodels"),
                 ("pandas", "pandas"), ("scipy", "scipy")]:
    try:
        importlib.import_module(mod)
    except ImportError:   # opencv-python と opencv-python-headless を両方入れると壊れることがあるので、cv2 が無いときだけ入れる
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import cv2, sklearn, statsmodels, scipy, numpy
print("OK", "OpenCV", cv2.__version__, "scikit-learn", sklearn.__version__, "statsmodels", statsmodels.__version__,
      "scipy", scipy.__version__, "numpy", numpy.__version__)
print("（教科書は OpenCV 5.1.0・scikit-learn 1.9.1・statsmodels 0.15.0 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の8つのセルで、8-6 のファイルを書き出します（`bo_sim.py`・`bo_next.py` は、同じフォルダの `bo_core.py` を読み込みます）。

In [ ]:
%%writefile drop_synth.py
#!/usr/bin/env python3
"""第8章：答えのわかった「合成の水滴画像」を作る（contact_angle.py の確かめ用）

使い方: cd ~/home-lab/ch8 && uv run python drop_synth.py --theta 30 70 110 --out synth
        uv run python drop_synth.py --theta 70 --model yl --volume-ul 5 --out synth
        uv run python drop_synth.py --coin --mm-per-px 0.025 --out synth   （スケール用の1円玉の影絵）

作る画像（スマホで横から撮った、背景を明るくした影絵を想定）
  - 明るい背景（上下に少し明るさのむら）、暗い水滴と、その下の暗い基板
  - 水滴の真ん中の明るい点（水滴がレンズになって背景の光を集める）
  - 傾き（--tilt 度）、ぼけ（--blur 画素）、雑音（--noise 階調）
  - --reflect を付けると、レンズが基板面より少し上にあるときの「基板に映った水滴」を描く
形のモデル
  - cap：球の一部（重力を無視。小さい滴のモデル）。--radius-px は接触半径（画素）
  - yl ：Young–Laplace の式を数値的に解いた形（重力で少しつぶれる）。--volume-ul と --mm-per-px で大きさが決まる
"""
import argparse
import json
import pathlib

import cv2
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

SS = 4                      # 4倍の細かさで描いてから縮める（ふちのなめらかさのため）
G, RHO, GAMMA = 9.80665, 997.0, 0.0720   # 重力加速度、水の密度 kg/m^3、表面張力 N/m（約25℃の目安）
LC = np.sqrt(GAMMA / (RHO * G))          # 毛管長 [m]（約2.7mm）


def cap_profile(theta_deg, r_px, n=400):
    """球の一部の右半分の輪郭。戻り値 (x, h)：x は中心からの横の距離、h は基線からの高さ（画素）"""
    th = np.radians(theta_deg)
    R = r_px / np.sin(th)                 # 球の半径
    hc = -R * np.cos(th)                  # 球の中心の高さ（基線から上を正）
    phi = np.linspace(0.0, th, n)         # 頂点からの角度
    return R * np.sin(phi), hc + R * np.cos(phi)


def yl_profile(theta_deg, volume_ul, n=400):
    """Young–Laplace の式（軸対称）を頂点から積分した右半分の輪郭 [m]。頂点の曲率半径 b を体積に合わせて決める"""
    th = np.radians(theta_deg)

    def shoot(b):
        def f(s, y):
            x, z, phi, v = y
            dphi = 1.0 / b if x < 1e-12 else 2.0 / b + z / LC**2 - np.sin(phi) / x
            return [np.cos(phi), np.sin(phi), dphi, np.pi * x * x * np.sin(phi)]
        ev = lambda s, y: y[2] - th
        ev.terminal = True
        return solve_ivp(f, [0, 50 * b + 0.05], [0, 0, 0, 0], events=ev,
                         max_step=b / 400, rtol=1e-9, atol=1e-12)

    vol = volume_ul * 1e-9
    b = brentq(lambda b: shoot(b).y[3, -1] - vol, 1e-5, 0.03)
    sol = shoot(b)
    x, z = sol.y[0], sol.y[1]
    idx = np.linspace(0, len(x) - 1, n).astype(int)
    return x[idx], (z[-1] - z)[idx], b


def render(prof_x, prof_h, args, rng):
    """影絵を描く。prof_x, prof_h は画素単位の右半分の輪郭"""
    W, H = args.width, args.height
    big = np.full((H * SS, W * SS), 1.0, np.float32)
    yy = np.linspace(-1, 1, H * SS)[:, None]
    big *= (args.bg - 15 * yy**2)                           # 背景の明るさのむら
    cx, yb = W / 2, args.baseline_frac * H                    # 水滴の中心と基線の高さ（画素）
    xs = np.r_[-prof_x[::-1], prof_x]
    hs = np.r_[prof_h[::-1], prof_h]
    pts = np.c_[cx + xs, yb - hs]
    s = 16                                                     # fillPoly の小数点以下ビット
    to_i = lambda p: np.round(p * SS * s).astype(np.int32)
    # 基板（基線より下）。--reflect のときは、基線のすぐ下に「映り込み」の帯を作る
    if args.reflect:
        band = int(args.reflect_px * SS)
        big[int(yb * SS):int(yb * SS) + band, :] = args.bg * 0.85
        mirror = np.c_[cx + xs, yb + hs * 0.9]                 # 斜めから見るので少し縮んで映る
        cv2.fillPoly(big, [to_i(mirror)], float(args.dark + 25), lineType=cv2.LINE_AA, shift=4)
        big[int(yb * SS) + band:, :] = args.dark
    else:
        big[int(round(yb * SS)):, :] = args.dark
    cv2.fillPoly(big, [to_i(pts)], float(args.dark), lineType=cv2.LINE_AA, shift=4)
    # 真ん中の明るい点（レンズ効果）
    hmax = prof_h.max()
    cv2.ellipse(big, (int(cx * SS), int((yb - 0.55 * hmax) * SS)),
                (int(0.35 * prof_x.max() * SS), int(0.18 * hmax * SS)), 0, 0, 360,
                float(args.bg * 0.9), -1, lineType=cv2.LINE_AA)
    # 傾き（画面全体を回す）
    M = cv2.getRotationMatrix2D((W * SS / 2, H * SS / 2), args.tilt, 1.0)
    big = cv2.warpAffine(big, M, (W * SS, H * SS), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)
    img = cv2.resize(big, (W, H), interpolation=cv2.INTER_AREA)
    if args.blur > 0:
        img = cv2.GaussianBlur(img, (0, 0), args.blur)
    img = img + rng.normal(0, args.noise, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--theta", type=float, nargs="+", default=[30, 70, 110])
    ap.add_argument("--model", choices=["cap", "yl"], default="cap")
    ap.add_argument("--radius-px", type=float, default=160, help="cap のときの接触半径（画素）")
    ap.add_argument("--volume-ul", type=float, default=3.0, help="yl のときの体積（µL）")
    ap.add_argument("--mm-per-px", type=float, default=None,
                    help="yl・coin のときの 1画素の大きさ（mm）。既定は yl で 0.008、--coin で 0.025")
    ap.add_argument("--tilt", type=float, default=1.5, help="画面の傾き（度）")
    ap.add_argument("--blur", type=float, default=1.5)
    ap.add_argument("--noise", type=float, default=4.0)
    ap.add_argument("--reflect", action="store_true")
    ap.add_argument("--reflect-px", type=float, default=60)
    ap.add_argument("--coin", action="store_true", help="直径20mm の円板（1円玉）の影絵を作る")
    ap.add_argument("--width", type=int, default=1200)
    ap.add_argument("--height", type=int, default=800)
    ap.add_argument("--baseline-frac", type=float, default=0.62)
    ap.add_argument("--bg", type=float, default=225)
    ap.add_argument("--dark", type=float, default=30)
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--out", default="synth")
    args = ap.parse_args()
    if args.mm_per_px is None:
        args.mm_per_px = 0.025 if args.coin else 0.008                         # 1円玉は 800画素に収まる大きさ
    out = pathlib.Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(args.seed)

    if args.coin:
        # 1円玉をつまようじに貼って台から浮かせた影絵。上の一部は画面からはみ出す
        d_px = 20.0 / args.mm_per_px
        W, H = args.width, args.height
        if d_px > 0.8 * W:
            raise SystemExit(f"1円玉の直径 {d_px:.0f} 画素が、画像の横幅 {W} 画素の8割を超えます。"
                             "--mm-per-px を大きくしてください（例：--mm-per-px 0.025）")
        big = np.full((H * SS, W * SS), args.bg, np.float32)
        yb = args.baseline_frac * H
        big[int(yb * SS):, :] = args.dark                                       # 台
        cy = yb - 30 - d_px / 2                                                  # 台から30画素浮かせる
        cv2.rectangle(big, (int((W / 2 - 4) * SS), int(cy * SS)), (int((W / 2 + 4) * SS), int(yb * SS)),
                      float(args.dark), -1)                                      # つまようじ
        cv2.circle(big, (int(W / 2 * SS), int(cy * SS)), int(d_px / 2 * SS), float(args.dark), -1,
                   lineType=cv2.LINE_AA)
        img = cv2.resize(big, (W, H), interpolation=cv2.INTER_AREA)
        img = cv2.GaussianBlur(img, (0, 0), args.blur) + rng.normal(0, args.noise, (H, W))
        cv2.imwrite(str(out / "coin.png"), np.clip(img, 0, 255).astype(np.uint8))
        print(f"{out/'coin.png'}  直径 {d_px:.1f} 画素 = 20.0 mm → {args.mm_per_px} mm/画素")
        return

    truth = {}
    for th in args.theta:
        if args.model == "cap":
            x, h = cap_profile(th, args.radius_px)
            info = {"theta": th, "model": "cap", "r_px": args.radius_px}
        else:
            xm, hm, b = yl_profile(th, args.volume_ul)
            x, h = xm * 1e3 / args.mm_per_px, hm * 1e3 / args.mm_per_px
            info = {"theta": th, "model": "yl", "volume_ul": args.volume_ul,
                    "mm_per_px": args.mm_per_px, "Bo": float((xm[-1] / LC) ** 2)}
        img = render(x, h, args, rng)
        name = f"{args.model}_{th:g}{'_refl' if args.reflect else ''}.png"
        cv2.imwrite(str(out / name), img)
        info.update(tilt=args.tilt, blur=args.blur, noise=args.noise, reflect=args.reflect)
        truth[name] = info
        print(name, json.dumps(info, ensure_ascii=False))
    path = out / "truth.json"
    old = json.loads(path.read_text()) if path.exists() else {}
    old.update(truth)
    path.write_text(json.dumps(old, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile contact_angle.py
#!/usr/bin/env python3
"""第8章：横から撮った水滴の写真から接触角を求める

使い方: cd ~/home-lab/ch8
  uv run python contact_angle.py photos/R01_D1.jpg photos/R01_D2.jpg --csv angles.csv
  uv run python contact_angle.py photos/*.jpg --csv angles.csv --mm-per-px 0.0081
  uv run python contact_angle.py photos/x.jpg --baseline mirror      （基板に水滴が映っているとき）
  uv run python contact_angle.py photos/x.jpg --baseline 512.3 518.9 （左端・右端の基線の y を手で与える）
  uv run python contact_angle.py --calib photos/coin.jpg --calib-mm 20.0   （スケールを求める）

前提：背景が明るく、水滴と基板が暗く写った影絵。水滴は画面の左右の真ん中あたり、
      基板は画面の左右の端まで写っている（--baseline auto のとき、左右の端で基板の上の縁を探す）。

やること
  1. 基線を決める（auto：画面の左右の端の基板の縁に直線を当てはめる。傾きもここで求める）
  2. 画像を回して基線を水平にし、OpenCV で水滴の輪郭（基線より上）を取り出す
  3. 各行で、明るさが「背景と水滴の中間」を横切る位置を小数の画素まで求める（サブピクセルの輪郭）
  4. 接触角を4通りで求める
       tan ：左右それぞれ、接触点の近く（基線から h_skip〜h_skip+fit 画素）の輪郭に小さな円を当てはめ、
             その円が基線と交わる点での接線の角度（局所的な円の当てはめによる接線法）
       ell ：輪郭に楕円を当てはめ（cv2.fitEllipse）、基線との交点での接線
       circ：輪郭に円を当てはめ、cosθ = −(中心の高さ)/半径
       half：θ/2 法  tan(θ/2) = 2H/D（球の一部と仮定）
  5. 結果を CSV に1行ずつ追記し、check/ に確認用の図を保存する
"""
import argparse
import csv
import pathlib
import sys

import cv2
import numpy as np
from scipy.optimize import least_squares
from scipy.signal import find_peaks
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
LC_MM = 2.71          # 水の毛管長 [mm]（約25℃の目安。表面張力の値で変わる）
FIELDS = ["file", "theta_tan", "theta_tan_L", "theta_tan_R", "theta_ell", "theta_circ", "theta_half",
          "lr_diff", "D_px", "H_px", "D_mm", "H_mm", "V_ul", "Bo", "tilt_deg", "baseline", "n_pts"]


def load_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        sys.exit(f"{path} を読めません（HEIC なら sips で JPEG にする：8-6 の run_ch8.sh 参照）")
    return img.astype(np.float32)


def levels(g):
    """背景（明るい）と影（暗い）の明るさ。上の15%の行の中央値と、暗い側の画素の中央値"""
    bg = float(np.median(g[: max(5, g.shape[0] * 15 // 100)]))
    dark = float(np.median(g[g < 0.5 * bg])) if np.any(g < 0.5 * bg) else 0.0
    return bg, dark


def crossing(prof, level, start, step):
    """1次元の明るさ prof で、start から step（+1/−1）の向きに進み、level を最初に下回る位置（小数）"""
    i = start
    n = len(prof)
    while 0 <= i + step < n:
        a, b = prof[i], prof[i + step]
        if a >= level > b:
            return i + step * (a - level) / (a - b)
        i += step
    return np.nan


def baseline_auto(g, mid, margin=0.08):
    """左右の端の帯（幅の margin ずつ）の各列で、上から見て最初に暗くなる位置を求め、直線 y = a x + c を当てはめる"""
    H, W = g.shape
    gs = cv2.GaussianBlur(g, (0, 0), 1.0)
    cols = list(range(2, int(W * margin))) + list(range(int(W * (1 - margin)), W - 2))
    xs, ys = [], []
    for x in cols:
        y = crossing(gs[:, x], mid, 0, +1)
        if np.isfinite(y):
            xs.append(x)
            ys.append(y)
    xs, ys = np.array(xs), np.array(ys)
    if len(xs) < 10:
        sys.exit("左右の端で基板の縁が見つかりません。基板を画面の端まで写すか、--baseline で y を与える")
    a, c = np.polyfit(xs, ys, 1)
    keep = np.abs(ys - (a * xs + c)) < 3 * max(1.0, np.median(np.abs(ys - (a * xs + c))) * 1.5)
    a, c = np.polyfit(xs[keep], ys[keep], 1)      # はずれた列を除いてもう一度
    return a, c


def rotate(g, a, c):
    """基線 y = a x + c が水平になるように回す。戻り値：回した画像、基線の行（小数）、角度（度）"""
    H, W = g.shape
    ang = np.degrees(np.arctan(a))
    M = cv2.getRotationMatrix2D((W / 2, H / 2), ang, 1.0)
    r = cv2.warpAffine(g, M, (W, H), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    p = M @ np.array([W / 2, a * W / 2 + c, 1.0])  # 基線上の1点を回した先
    return r, float(p[1]), ang


def drop_mask(r, yb, mid):
    """基線より上の暗い部分（OpenCV の輪郭）。いちばん大きく、基線に接するもの。中の明るい点は埋める"""
    H, W = r.shape
    m = (r < mid).astype(np.uint8)
    m[int(np.floor(yb)) - 1:, :] = 0                  # 基線より下は使わない
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cand = [c for c in cnts if c[:, 0, 1].max() >= int(np.floor(yb)) - 3]
    if not cand:
        sys.exit("基線に接する暗い部分（水滴）が見つかりません")
    c = max(cand, key=cv2.contourArea)
    filled = np.zeros_like(m)
    cv2.drawContours(filled, [c], -1, 1, thickness=-1)   # 外側の輪郭の内側を全部塗る（明るい点を埋める）
    return filled, c


def subpixel_edges(r, mask, yb, mid):
    """各行（基線より上）で、左右のふちを小数の画素で求める。戻り値：h, xL, xR（h は基線からの高さ）"""
    gs = cv2.GaussianBlur(r, (0, 0), 0.7)
    rows = np.where(mask.any(axis=1))[0]
    hs, xl, xr = [], [], []
    for y in rows:
        xs = np.where(mask[y])[0]
        x0, x1 = xs.min(), xs.max()
        L = crossing(gs[y], mid, max(0, x0 - 6), +1)
        R = crossing(gs[y], mid, min(r.shape[1] - 1, x1 + 6), -1)
        if np.isfinite(L) and np.isfinite(R) and R > L:
            hs.append(yb - y)
            xl.append(L)
            xr.append(R)
    return np.array(hs), np.array(xl), np.array(xr)


def baseline_mirror(g, mid):
    """基板に水滴が映っているとき：水滴とその映り込みの輪郭が折れ曲がる高さを基線とする（傾きは基板の手前の縁で直したあと）
    θ < 90° なら幅がいちばん広い行、θ > 90° ならくびれの行の近くで、上下の輪郭の2次式の交点を探す。
    θ が 90° に近いと輪郭がほぼまっすぐつながって折れ曲がりが見えないので、measure() で結果を使わない"""
    gs = cv2.GaussianBlur(g, (0, 0), 1.0)
    m = (gs < mid).astype(np.uint8)
    H, W = g.shape
    cx = W // 2
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max((c for c in cnts if cv2.pointPolygonTest(c, (cx, float(np.argmax(m[:, cx]) + 3)), False) >= 0),
            key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    ys, wl, wr = [], [], []
    for yy in range(y, y + h):
        L = crossing(gs[yy], mid, max(0, x - 6), +1)
        R = crossing(gs[yy], mid, min(W - 1, x + w + 6), -1)
        if np.isfinite(L) and np.isfinite(R) and R - L < 0.95 * W:   # 基板の帯の行は除く
            ys.append(yy)
            wl.append(L)
            wr.append(R)
    ys, wl, wr = map(np.array, (ys, wl, wr))
    width = wr - wl
    # 幅の山を探す。山が2つ（水滴の赤道とその映り込みの赤道。θ>90°）なら間のくびれ、
    # 山が1つ（θ<90°）ならその山の行を、粗い基線にする
    k = np.convolve(width, np.ones(9) / 9, mode="same")
    peaks, prop = find_peaks(k, prominence=0.01 * k.max())
    if len(peaks) >= 2:
        p1, p2 = sorted(peaks[np.argsort(prop["prominences"])[-2:]])
        i0 = p1 + int(np.argmin(k[p1:p2 + 1]))
    else:
        i0 = int(peaks[0]) if len(peaks) else int(np.argmax(k))
    y0 = ys[i0]
    # 折れ曲がりの位置を小数で：上側と下側にそれぞれ2次式を当てはめ、交わる高さ
    est = []
    for e in (wl, wr):
        up = (ys > y0 - 25) & (ys < y0 - 4)
        dn = (ys > y0 + 4) & (ys < y0 + 25)
        if up.sum() < 5 or dn.sum() < 5:
            continue
        pu, pd = np.polyfit(ys[up], e[up], 2), np.polyfit(ys[dn], e[dn], 2)
        roots = np.roots(pu - pd)
        roots = roots[np.isreal(roots)].real
        if len(roots):
            est.append(roots[np.argmin(np.abs(roots - y0))])
    yb = float(np.mean(est)) if est else float(y0)
    return 0.0, yb


def kasa(x, y):
    """代数的な円の当てはめ（速いが、短い円弧では少しかたよる）"""
    A = np.c_[2 * x, 2 * y, np.ones_like(x)]
    c = np.linalg.lstsq(A, x**2 + y**2, rcond=None)[0]
    return c[0], c[1], float(np.sqrt(max(c[2] + c[0] ** 2 + c[1] ** 2, 1e-12)))


def circle_intersections(c1, c2):
    """2つの円 (xc, yc, R) の交点のリスト（交わらなければ空）"""
    (x1, y1, r1), (x2, y2, r2) = c1, c2
    d = np.hypot(x2 - x1, y2 - y1)
    if d == 0 or d > r1 + r2 or d < abs(r1 - r2):
        return []
    a = (r1**2 - r2**2 + d**2) / (2 * d)
    h = np.sqrt(max(r1**2 - a**2, 0.0))
    xm, ym = x1 + a * (x2 - x1) / d, y1 + a * (y2 - y1) / d
    return [(xm + h * (y2 - y1) / d, ym - h * (x2 - x1) / d), (xm - h * (y2 - y1) / d, ym + h * (x2 - x1) / d)]


def fit_circle(x, y):
    """円の当てはめ。代数的な方法（Kasa 法）で初期値を出し、点から円までの距離の2乗和を最小にする（幾何学的な当てはめ）"""
    p0 = list(kasa(x, y))
    res = least_squares(lambda p: np.hypot(x - p[0], y - p[1]) - p[2], p0)
    return float(res.x[0]), float(res.x[1]), float(abs(res.x[2]))


def ellipse_angles(x, h):
    """楕円を当てはめ、基線（h=0）との2つの交点での接触角（左・右）を返す"""
    pts = np.c_[x, -h].astype(np.float32)          # OpenCV の座標（y は下向き）に合わせる
    (cx, cy), (A, B), ang = cv2.fitEllipse(pts)
    a, b, t0 = A / 2, B / 2, np.radians(ang)
    t = np.linspace(0, 2 * np.pi, 20001)
    px = cx + a * np.cos(t) * np.cos(t0) - b * np.sin(t) * np.sin(t0)
    py = cy + a * np.cos(t) * np.sin(t0) + b * np.sin(t) * np.cos(t0)
    s = np.sign(py)                                  # 基線は py = 0
    idx = np.where(s[:-1] != s[1:])[0]
    if len(idx) < 2:
        return np.nan, np.nan
    out = {}
    for i in idx:
        f = py[i] / (py[i] - py[i + 1])
        ti = t[i] + f * (t[i + 1] - t[i])
        xi = cx + a * np.cos(ti) * np.cos(t0) - b * np.sin(ti) * np.sin(t0)
        dx = -a * np.sin(ti) * np.cos(t0) - b * np.cos(ti) * np.sin(t0)
        dy = -a * np.sin(ti) * np.sin(t0) + b * np.cos(ti) * np.cos(t0)
        if dy > 0:                                   # 上向き（OpenCV の y が減る向き）にそろえる
            dx, dy = -dx, -dy
        out[xi] = (dx, -dy)                          # (横, 上向きの成分)
    xs = sorted(out)
    (dxl, dul), (dxr, dur) = out[xs[0]], out[xs[-1]]
    return np.degrees(np.arctan2(dul, dxl)), np.degrees(np.arctan2(dur, -dxr))


def measure(path, args, mm_per_px=None, save_check=True):
    g = load_gray(path)
    bg, dark = levels(g)
    mid = 0.5 * (bg + dark)
    if args.baseline == ["auto"]:
        a, c = baseline_auto(g, mid)
        r, yb, tilt = rotate(g, a, c)
    elif args.baseline == ["mirror"]:
        a, c = baseline_auto(g, mid)                 # 傾きだけは基板の手前の縁から求める
        r, _, tilt = rotate(g, a, c)
        _, yb = baseline_mirror(r, mid)
    else:
        yl, yr = map(float, args.baseline)
        W = g.shape[1]
        a = (yr - yl) / (W - 1)
        r, yb, tilt = rotate(g, a, yl)
    mask, cnt = drop_mask(r, yb, mid)
    h, xl, xr = subpixel_edges(r, mask, yb, mid)
    if len(h) < 10:
        sys.exit(f"{path}: 輪郭の点が少なすぎます（水滴が小さい・薄い、ピントが合っていない）")
    Hpx = float(h.max())
    # 頂点の高さを小数で：真ん中の列を上から見て暗くなる位置
    xc0 = 0.5 * (xl[np.argmax(h)] + xr[np.argmax(h)])
    ytop = crossing(cv2.GaussianBlur(r, (0, 0), 0.7)[:, int(round(xc0))], mid, 0, +1)
    if np.isfinite(ytop):
        Hpx = yb - ytop
    skip = args.h_skip
    fit = min(max(args.fit_px, 0.25 * Hpx), Hpx - skip)   # 薄い滴（θ が小さい）では、ほぼ全体を使うことになる
    sel = (h > skip) & (h <= skip + fit)
    if sel.sum() < 5:
        sys.exit(f"{path}: 接触点の近くの点が少なすぎます（--h-skip・--fit-px を見直す）")
    sides = {}
    for name, xe in (("L", xl), ("R", xr)):
        cx, ch, cr = fit_circle(xe[sel], h[sel])
        th = np.degrees(np.arccos(np.clip(-ch / cr, -1, 1)))      # 液体は円の内側
        dx = np.sqrt(max(cr**2 - ch**2, 0.0))
        sides[name] = (th, cx - dx if name == "L" else cx + dx, (cx, ch, cr))
    (th_l, x_cl, cl), (th_r, x_cr, cr_) = sides["L"], sides["R"]
    D = x_cr - x_cl
    xm = 0.5 * (x_cl + x_cr)
    use = h > skip
    X = np.r_[xl[use], xr[use]] - xm
    Y = np.r_[h[use], h[use]]
    ell_l, ell_r = ellipse_angles(X, Y)
    xc, yc, R = fit_circle(X, Y)
    th_circ = np.degrees(np.arccos(np.clip(-yc / R, -1, 1)))
    th_tan = 0.5 * (th_l + th_r)
    if args.baseline == ["mirror"] and (85 <= th_tan <= 100 or 85 <= th_circ <= 100 or abs(th_tan - th_circ) > 2):
        # θ が 90° 付近だと水滴と映り込みの輪郭がほぼまっすぐつながり、折れ曲がりから基線が決まらない
        # （合成画像で 90〜98° のとき、基線が数〜10画素ずれて θ が数度ずれた）
        sys.exit(f"{path}: --baseline mirror では θ が 90° 付近（接線 {th_tan:.1f}°・円 {th_circ:.1f}°）の基線が決まりません。"
                 "レンズの高さを基板の面に合わせて映り込みを消し --baseline auto にするか、--baseline で y を手で与える")
    th_half = np.degrees(2 * np.arctan(2 * Hpx / D))
    # 体積（各行の幅を直径とする円板の和）。h<skip の部分は D の円柱で近似
    w = xr - xl
    V_px = np.sum(np.pi / 4 * w[use] ** 2) + np.pi / 4 * D**2 * skip
    row = {"file": pathlib.Path(path).name,
           "theta_tan": 0.5 * (th_l + th_r), "theta_tan_L": th_l, "theta_tan_R": th_r,
           "theta_ell": 0.5 * (ell_l + ell_r), "theta_circ": th_circ, "theta_half": th_half,
           "lr_diff": th_l - th_r, "D_px": D, "H_px": Hpx, "tilt_deg": tilt,
           "baseline": " ".join(args.baseline), "n_pts": int(use.sum())}
    if mm_per_px:
        row["D_mm"] = D * mm_per_px
        row["H_mm"] = Hpx * mm_per_px
        row["V_ul"] = V_px * mm_per_px**3            # 1 mm^3 = 1 µL
        row["Bo"] = (D * mm_per_px / 2 / LC_MM) ** 2   # 接触半径で定義したボンド数
    if save_check:
        check_plot(path, r, yb, h, xl, xr, sel, cl, cr_, (xc + xm, yc, R), row, args.check_dir)
    return row


def check_plot(path, r, yb, h, xl, xr, sel, cl, cr, circ, row, outdir):
    out = pathlib.Path(outdir)
    out.mkdir(exist_ok=True)
    pad = 40
    x0 = int(max(0, xl.min() - pad)); x1 = int(min(r.shape[1], xr.max() + pad))
    y0 = int(max(0, yb - h.max() - pad)); y1 = int(min(r.shape[0], yb + pad))
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(r[y0:y1, x0:x1], cmap="gray", extent=(x0, x1, y1, y0))
    ax.plot(xl, yb - h, ".", ms=1.5, color="tab:orange", label="輪郭（サブピクセル）")
    ax.plot(xr, yb - h, ".", ms=1.5, color="tab:orange")
    ax.axhline(yb, color="tab:cyan", lw=1, label="基線")
    t = np.linspace(0, 2 * np.pi, 400)
    for i, (cx, ch, rr) in enumerate((cl, cr)):
        # 接触点での接線（長さ 0.6×当てはめの高さ）
        side = -1 if i == 0 else 1
        dx = np.sqrt(max(rr**2 - ch**2, 0.0))
        xc = cx + side * dx
        tx, ty = side * ch, dx                  # 円の接線の向き（半径に垂直で、上向き）
        L = 1.5 * h[sel].max() / max(np.hypot(tx, ty), 1e-9)
        ax.plot([xc, xc + tx * L], [yb, yb - ty * L], color="tab:red", lw=1.5,
                label="接線（局所円）" if i == 0 else None)
    ax.plot(circ[0] + circ[2] * np.cos(t), yb - (circ[1] + circ[2] * np.sin(t)), "--", color="tab:green",
            lw=1, label="円の当てはめ（全体）")
    ax.set_xlim(x0, x1); ax.set_ylim(y1, y0)
    ax.set_title(f"{row['file']}  接線 {row['theta_tan']:.1f}°（左 {row['theta_tan_L']:.1f}・右 {row['theta_tan_R']:.1f}）"
                 f"  楕円 {row['theta_ell']:.1f}°  円 {row['theta_circ']:.1f}°", fontsize=9)
    ax.legend(fontsize=8, loc="upper right")
    fig.tight_layout()
    fig.savefig(out / (pathlib.Path(path).stem + "_check.png"), dpi=120)
    plt.close(fig)


def calibrate(path, mm):
    """1円玉など、直径が分かっている円板の影から mm/画素 を求める。
    円板は台から浮かせて（つまようじに貼るなど）撮る。画面からはみ出した部分や、つまようじの影は
    「円から外れた点」として取り除き、残った円弧に円を当てはめる"""
    g = load_gray(path)
    bg, dark = np.percentile(g, 95), np.percentile(g, 5)
    H, W = g.shape
    m = (cv2.GaussianBlur(g, (0, 0), 1.0) < 0.5 * (bg + dark)).astype(np.uint8)
    m[m.mean(axis=1) > 0.97] = 0                    # 横いっぱいに暗い行（台）は使わない
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        sys.exit("円板の影が見つかりません（背景を明るく、円板を画面の横幅より小さく写す）")
    c = max(cnts, key=cv2.contourArea)[:, 0, :].astype(float)
    inner = (c[:, 0] > 2) & (c[:, 0] < W - 3) & (c[:, 1] > 2) & (c[:, 1] < H - 3)   # 画面のふちの点は使わない
    x, y = c[inner, 0], c[inner, 1]
    keep = np.ones(len(x), bool)
    for _ in range(5):                                # 円から外れた点（つまようじ・台の影）を除いて当てはめ直す
        xc, yc, R = fit_circle(x[keep], y[keep])
        res = np.abs(np.hypot(x - xc, y - yc) - R)
        keep = res < max(2.0, 3 * np.median(res[keep]))
    xc, yc, R = fit_circle(x[keep], y[keep])
    rms = float(np.sqrt(np.mean((np.hypot(x[keep] - xc, y[keep] - yc) - R) ** 2)))
    ang = np.sort(np.arctan2(y[keep] - yc, x[keep] - xc))
    span = 360 - np.degrees(np.max(np.diff(np.r_[ang, ang[0] + 2 * np.pi])))   # 円弧の見えている角度
    d = 2 * R + 1.0           # findContours は境界の画素の中心をたどるので、直径が約1画素小さく出る分を足す
    print(f"円板の直径 {d:.1f} 画素（使った点 {keep.sum()} / {len(x)}、円からのずれ RMS {rms:.2f} 画素、見えている円弧 {span:.0f}°）")
    if span < 120:
        print("  注意：見えている円弧が短い。円板がもっと画面に入るように撮る")
    if rms > 1.0:
        print("  注意：円からのずれが大きい。円板がカメラに正対していない・ピンぼけ・台の影とつながっている可能性")
    print(f"→ {mm / d:.6f} mm/画素（--mm-per-px にこの値を渡す）")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("images", nargs="*")
    ap.add_argument("--csv", default="angles.csv")
    ap.add_argument("--baseline", nargs="+", default=["auto"], help="auto / mirror / 左のy 右のy")
    ap.add_argument("--h-skip", type=float, default=4.0, help="基線のすぐ上の、使わない高さ（画素）")
    ap.add_argument("--fit-px", type=float, default=15.0, help="接線の当てはめに使う高さ（画素）の最小値。実際は max(これ, 0.25×水滴の高さ)")
    ap.add_argument("--mm-per-px", type=float, default=None)
    ap.add_argument("--check-dir", default="check")
    ap.add_argument("--calib", default=None)
    ap.add_argument("--calib-mm", type=float, default=20.0)
    args = ap.parse_args()
    if args.calib:
        calibrate(args.calib, args.calib_mm)
        return
    if not args.images:
        ap.error("画像を1つ以上指定してください")
    path = pathlib.Path(args.csv)
    new = not path.exists()
    with open(path, "a", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            wr.writeheader()
        for p in args.images:
            try:
                row = measure(p, args, args.mm_per_px)
            except SystemExit as e:               # 1枚うまくいかなくても、残りは続ける
                print(f"とばした：{e}")
                continue
            wr.writerow({k: (f"{v:.4f}" if isinstance(v, float) else v) for k, v in row.items()})
            print(f"{row['file']}: 接線 {row['theta_tan']:.1f}°（左右の差 {row['lr_diff']:+.1f}°） 楕円 {row['theta_ell']:.1f}°"
                  f"  円 {row['theta_circ']:.1f}°  θ/2 {row['theta_half']:.1f}°  傾き {row['tilt_deg']:+.2f}°"
                  + (f"  体積 {row['V_ul']:.2f} µL  Bo {row['Bo']:.2f}" if args.mm_per_px else ""))
            if abs(row["lr_diff"]) > 3:
                print("  注意：左右の接触角が3°以上違う（台の傾き、基線のずれ、試料の汚れ・傷を確かめる）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile design_ch8.py
#!/usr/bin/env python3
"""第8章：2^3 要因計画（乱塊法）の実験表を作り、順番と試料片の割り当てを無作為化する

使い方: cd ~/home-lab/ch8 && uv run python design_ch8.py --blocks 2 --drops 3 --out design_ch8.csv
        （--seed を省くと毎回ちがう乱数の種を使い、その値を表示する。表示された種はノートに書く）

要因と水準（符号 −1 / +1）
  A 素材      ：PP（リサイクルマーク 5） / PET（リサイクルマーク 1）
  B 紙やすり  ：なし / #400 で一方向に10往復
  C アルコール：なし / 消毒用エタノール（または IPA）でふいて乾かす
ブロック：日（1日に8条件を1回ずつ。--blocks 日分）。日の中で順番を無作為にする
試料片  ：素材ごとに P1, P2, … / E1, E2, … の印をつけた片を、条件に無作為に割り当てる
1つの試料片に --drops 個の滴をのせ、写真の名前を R<実験番号>_D<滴の番号>.jpg にする
"""
import argparse
import itertools
import pathlib

import numpy as np
import pandas as pd

LEVELS = {"A": ("PP", "PET"), "B": ("なし", "#400"), "C": ("なし", "アルコール")}
NAMES = {"A": "material", "B": "sanding", "C": "cleaning"}


def l8_table():
    """直交表 L8(2^7)。列1・2・4 に A・B・C を割り付けると、残りの列が交互作用になる"""
    base = np.array(list(itertools.product([-1, 1], repeat=3)))   # A, B, C
    A, B, C = base[:, 0], base[:, 1], base[:, 2]
    cols = {"1:A": A, "2:B": B, "3:AB": A * B, "4:C": C, "5:AC": A * C, "6:BC": B * C, "7:ABC": A * B * C}
    return pd.DataFrame(cols)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--blocks", type=int, default=2, help="ブロック（日）の数")
    ap.add_argument("--drops", type=int, default=3, help="1つの試料片にのせる滴の数")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out", default="design_ch8.csv")
    args = ap.parse_args()
    seed = args.seed if args.seed is not None else int(np.random.SeedSequence().entropy % 100000)
    rng = np.random.default_rng(seed)
    print(f"乱数の種 seed = {seed}（ノートに書く。同じ表を作り直すときは --seed {seed}）\n")

    l8 = l8_table()
    print("直交表 L8(2^7) と割り付け（−1/+1。行は標準の順番）")
    print(l8.to_string(index=False))
    print("各列は互いに直交している（どの2列も、(−1,−1)(−1,+1)(+1,−1)(+1,+1) が2回ずつ）。")
    print("4つ目の要因 D を列7 に入れると 2^(4−1) の一部実施になり、D と A×B×C が区別できなくなる（交絡）。\n")

    rows = []
    n_piece = {"PP": 0, "PET": 0}
    piece_ids = {m: list(rng.permutation(args.blocks * 4) + 1) for m in ("PP", "PET")}
    run = 0
    for blk in range(1, args.blocks + 1):
        order = rng.permutation(len(l8))
        for k, i in enumerate(order, start=1):
            run += 1
            a, b, c = int(l8.iloc[i]["1:A"]), int(l8.iloc[i]["2:B"]), int(l8.iloc[i]["4:C"])
            mat = LEVELS["A"][(a + 1) // 2]
            pid = piece_ids[mat][n_piece[mat]]
            n_piece[mat] += 1
            rows.append({"run": run, "block": blk, "order": k, "A": a, "B": b, "C": c,
                         "material": mat, "sanding": LEVELS["B"][(b + 1) // 2],
                         "cleaning": LEVELS["C"][(c + 1) // 2],
                         "piece": ("P" if mat == "PP" else "E") + str(pid),
                         "photos": " ".join(f"R{run:02d}_D{d}" for d in range(1, args.drops + 1))})
    df = pd.DataFrame(rows)
    out = pathlib.Path(args.out)
    if out.exists():
        raise SystemExit(f"{out} はもうあります。上書きしないので、消すか --out で別の名前にする")
    df.to_csv(out, index=False)
    print(f"実験表（{len(df)} 回 = 8条件 × {args.blocks} ブロック、各 {args.drops} 滴）→ {out}")
    print(df.drop(columns=["A", "B", "C"]).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile doe_anova.py
#!/usr/bin/env python3
"""第8章：2^3 要因計画（乱塊法）の分散分析。疑似反復と、交互作用を無視した主効果の読み違いを確かめる

使い方: cd ~/home-lab/ch8
  uv run python doe_anova.py design_ch8.csv angles.csv                 （自分の測定）
  uv run python doe_anova.py design_ch8.csv --demo demo_angles.csv     （合成データを作って練習）
  --col theta_ell のように、使う接触角の列を変えられる（既定は theta_tan）

やること
  1. angles.csv の file（R<実験番号>_D<滴>）と実験表を結びつけ、試料片ごとの平均を出す
  2. 【正しい】試料片を単位にした分散分析：theta ~ C(block) + A*B*C（A・B・C は −1/+1）
  3. 【誤り】滴を単位にした分散分析（同じ試料片の滴を独立とみなす＝疑似反復）
  4. 滴を単位にするなら、試料片を変量効果にした線形混合モデル（2 とほぼ同じ結論になる）
  5. 効果（+1 と −1 の差）と 95% 区間、ホルム法で補正した p 値、素材ごとの紙やすりの単純効果
  6. doe_effects.png：交互作用図と効果の区間
"""
import argparse
import re
import sys

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
TERMS = ["xA", "xB", "xC", "xA:xB", "xA:xC", "xB:xC", "xA:xB:xC"]   # C() と名前がぶつからないよう x を付ける
LABEL = {t: t.replace("x", "").replace(":", "×") for t in TERMS}
FORMULA = "theta ~ C(block) + xA*xB*xC"


def make_demo(design, path, drops=3, seed=0):
    """合成データ：PP 約100°・PET 約75°、紙やすりは PP で上げ PET で下げる（Wenzel の式の向き）"""
    rng = np.random.default_rng(seed)
    rows = []
    for _, r in design.iterrows():
        a, b, c = r["A"], r["B"], r["C"]
        mu = 87.5 - 12.5 * a + 0.0 * b - 5.0 * a * b - 1.5 * c + 2.0 * (r["block"] - 1)
        piece = rng.normal(0, 2.5)                       # 試料片ごとのばらつき
        for d in range(1, drops + 1):
            th = mu + piece + rng.normal(0, 2.0)         # 同じ試料片の上の滴どうしのばらつき
            rows.append({"file": f"R{int(r['run']):02d}_D{d}.jpg", "theta_tan": th})
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"合成データ → {path}（真の効果：A −25°、B 0°、A×B −10°、C −3°、ブロック +2°）\n")


def effects_table(fit, label):
    """係数（−1/+1 の符号での傾き）の2倍が効果（+1 の平均 − −1 の平均）"""
    ci = fit.conf_int()
    out = pd.DataFrame({"効果": 2 * fit.params[TERMS], "下限": 2 * ci.loc[TERMS, 0],
                        "上限": 2 * ci.loc[TERMS, 1], "p": fit.pvalues[TERMS]}).rename(index=LABEL)
    out["p_Holm"] = multipletests(out["p"], method="holm")[1]
    print(f"--- {label}：効果と95%区間（自由度 {int(fit.df_resid)}）")
    print(out.round(3).to_string())
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("design")
    ap.add_argument("angles", nargs="?")
    ap.add_argument("--demo", default=None, help="合成データを作ってこのファイルに保存し、それを解析する")
    ap.add_argument("--col", default="theta_tan")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    design = pd.read_csv(args.design)
    if args.demo:
        make_demo(design, args.demo, seed=args.seed)
        args.angles = args.demo
    if not args.angles:
        sys.exit("angles.csv を指定するか、--demo を付ける")
    ang = pd.read_csv(args.angles)
    m = ang["file"].str.extract(r"R(\d+)_D(\d+)", flags=re.I)
    ang["run"], ang["drop"] = pd.to_numeric(m[0]), pd.to_numeric(m[1])
    ang = ang.dropna(subset=["run"]).drop_duplicates("file", keep="last")   # 同じ写真を2回測ったら後のほう
    df = ang.merge(design, on="run", how="inner")
    df["theta"] = df[args.col]
    df["xA"], df["xB"], df["xC"] = df["A"], df["B"], df["C"]
    df = df.drop(columns=["A", "B", "C"])
    cnt = df.groupby("run").size()
    print(f"滴の数：{len(df)}（試料片 {cnt.size} 枚、1枚あたり {cnt.min()}〜{cnt.max()} 滴）")
    missing = sorted(set(design.run) - set(cnt.index))
    if missing:
        print(f"注意：写真のない実験番号 {missing}。分散分析は釣り合いが崩れる（型IIの平方和で計算する）")

    # 1) 試料片ごとの平均（これが独立な単位）
    sp = df.groupby(["run", "block", "xA", "xB", "xC", "material", "sanding", "cleaning", "piece"],
                    as_index=False)["theta"].mean()
    print("\n条件ごとの平均（試料片の平均の平均）")
    print(sp.groupby(["material", "sanding", "cleaning"])["theta"].agg(["mean", "std", "count"]).round(1))

    # 2) 正しい：試料片を単位にした分散分析
    f_ok = smf.ols(FORMULA, data=sp).fit()
    print("\n=== 【正しい】試料片を単位にした分散分析（型II）")
    print(anova_lm(f_ok, typ=2).rename(index=LABEL).round(4))
    e_ok = effects_table(f_ok, "試料片が単位")

    # 3) 誤り：滴を単位にした分散分析（疑似反復）
    f_bad = smf.ols(FORMULA, data=df).fit()
    print("\n=== 【誤り】滴を単位にした分散分析（同じ試料片の滴を独立とみなしている）")
    print(anova_lm(f_bad, typ=2).rename(index=LABEL).round(4))
    e_bad = effects_table(f_bad, "滴が単位（疑似反復）")
    print("→ 効果の推定値はほぼ同じでも、自由度が水増しされ、区間が狭く p が小さく出る。")
    print("  ばらつきの大部分が『試料片ごとの違い』なら、滴を増やしても試料片の違いは平均で消えない。")

    # 4) 滴を単位にするなら、試料片を変量効果に
    try:
        mm = smf.mixedlm(FORMULA, data=df, groups=df["run"]).fit(reml=True)
        se = mm.bse[TERMS]
        # 試料片が少ないので、正規分布の 1.96 ではなく t 分布（自由度＝試料片の数 − 固定効果の数）の値を使う
        dfm = int(f_ok.df_resid)
        q = stats.t.ppf(0.975, dfm)
        print(f"\n=== 線形混合モデル（試料片を変量効果）：効果 ± {q:.2f}×標準誤差（t 分布、自由度 {dfm}）")
        print(pd.DataFrame({"効果": 2 * mm.params[TERMS], "±": 2 * q * se,
                            "±（1.96 を使うと）": 2 * 1.96 * se}).rename(index=LABEL).round(2).to_string())
        print("  → 試料片が16枚しかないので、1.96 を使うと区間が狭すぎる（自由度の小さい t 分布の値を使う）")
        var_piece, var_drop = float(mm.cov_re.iloc[0, 0]), float(mm.scale)
        print(f"分散成分：試料片 {var_piece:.2f}、滴 {var_drop:.2f}（級内相関 ICC = {var_piece/(var_piece+var_drop):.2f}）")
    except Exception as e:            # 釣り合いが悪いと収束しないことがある
        print("線形混合モデルは計算できなかった：", e)

    # 5) 交互作用があるときの主効果の読み方
    print("\n=== 素材ごとの紙やすりの効果（単純効果、試料片が単位）")
    for mat, g in sp.groupby("material"):
        f = smf.ols("theta ~ C(block) + xB + xC", data=g).fit()
        lo, hi = 2 * f.conf_int().loc["xB"]
        print(f"  {mat}: {2*f.params['xB']:+.1f}°  95%区間 [{lo:+.1f}, {hi:+.1f}]（自由度 {int(f.df_resid)}）")
    print("  （素材ごとに分けて当てはめたので、残差の自由度は 8枚 − 4 = 4。全体のモデル（自由度7）の残差を使う方法もある）")
    print("→ B（紙やすり）の主効果が小さくても、『紙やすりは効かない』とは言えない。")
    print("  A×B の交互作用が大きいときは、主効果ではなく素材ごとの効果で語る。")

    # 6) 図
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    for mat, mk in (("PP", "o"), ("PET", "s")):
        g = sp[sp.material == mat]
        mean = g.groupby("xB")["theta"].mean()
        ax[0].plot([-1, 1], mean.values, marker=mk, ms=8, label=mat)
        ax[0].scatter(g.xB + np.where(g.xC > 0, 0.05, -0.05), g.theta, s=14, alpha=0.5)
    ax[0].set_xticks([-1, 1], ["紙やすり なし", "#400"])
    ax[0].set_ylabel("接触角 [°]（試料片の平均）")
    ax[0].set_title("交互作用図（線が平行でなければ交互作用）")
    ax[0].legend()
    y = np.arange(len(TERMS))
    for k, (e, lab, off) in enumerate(((e_ok, "試料片が単位（正しい）", -0.15), (e_bad, "滴が単位（疑似反復）", 0.15))):
        ax[1].errorbar(e["効果"], y + off, xerr=[e["効果"] - e["下限"], e["上限"] - e["効果"]],
                       fmt="o", capsize=3, label=lab)
    ax[1].axvline(0, color="gray", lw=0.8)
    ax[1].set_yticks(y, [LABEL[t] for t in TERMS])
    ax[1].invert_yaxis()
    ax[1].set_xlabel("効果 [°]（+1 の平均 − −1 の平均）と95%区間")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("doe_effects.png", dpi=130)
    print("\n図 → doe_effects.png")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile recovery_fit.py
#!/usr/bin/env python3
"""第8章：処理のあとの接触角の時間変化（疎水性回復）を、指数モデルの非線形回帰と状態空間モデルで当てはめる

使い方: cd ~/home-lab/ch8
  uv run python recovery_fit.py angles_rec.csv                    （自分の測定。写真の名前は G1_0030m_D2.jpg の形）
  uv run python recovery_fit.py --demo demo_rec.csv               （合成データを作って練習）
  --cut 60 を付けると、60分までのデータだけで当てはめたときの「外挿」の危うさも表示する

モデル
  非線形回帰：theta(t) = th_inf − (th_inf − th0)·exp(−t/tau)   （試料ごとに当てはめる）
  状態空間  ：x_k = th_inf + (x_{k−1} − th_inf)·exp(−Δt_k/tau) + w_k   （w：湿度などによる揺らぎ。OU 過程）
              揺らぎが平均の曲線に戻る時定数も、回復の tau と同じと仮定している（パラメータを減らすための単純化）
              y_k = x_k + v_k                                          （v：滴どうしのばらつき。分散は同じ時刻の滴から求める）
              カルマンフィルタで対数尤度を計算し、全試料で th_inf・tau・th0・q を共通として最尤推定
独立な単位は「試料（ガラス片など）」。同じ時刻の滴は平均してから使う（1つの試料の上の滴は疑似反復）。
"""
import argparse
import re
import sys

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit, minimize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def model(t, th_inf, th0, log_tau):
    return th_inf - (th_inf - th0) * np.exp(-t / np.exp(log_tau))


def make_demo(path, seed=0):
    """合成データ：3枚のガラス片。th0 約15°、th_inf 約45°、tau 約300分（試料ごとに少し違う）"""
    rng = np.random.default_rng(seed)
    times = [2, 5, 10, 20, 40, 60, 120, 240, 480, 1440, 2880]
    rows = []
    for s in ("G1", "G2", "G3"):
        th_inf, th0 = 45 + rng.normal(0, 3), 15 + rng.normal(0, 2)
        tau = 300 * np.exp(rng.normal(0, 0.25))
        dev = 0.0
        for k, t in enumerate(times):
            dev = 0.6 * dev + rng.normal(0, 1.2)        # その日の湿度などによる、ゆっくりした揺らぎ
            mu = model(t, th_inf, th0, np.log(tau)) + dev
            for d in (1, 2, 3):
                rows.append({"file": f"{s}_{t:04d}m_D{d}.jpg", "theta_tan": mu + rng.normal(0, 2.0)})
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"合成データ → {path}\n")


def load(path, col):
    a = pd.read_csv(path)
    m = a["file"].str.extract(r"([A-Za-z0-9]+)_(\d+)m_D(\d+)")
    a["spec"], a["t"], a["drop"] = m[0], pd.to_numeric(m[1]), pd.to_numeric(m[2])
    a = a.dropna(subset=["t"]).drop_duplicates("file", keep="last")
    a["theta"] = a[col]
    g = a.groupby(["spec", "t"])["theta"].agg(["mean", "var", "count"]).reset_index()
    return a, g


def fit_nls(t, y, p0=None):
    p0 = p0 or [y[-1], y[0], np.log(max(t.max() / 5, 1.0))]
    p, cov = curve_fit(model, t, y, p0=p0, maxfev=20000)
    return p, cov


def kalman_nll(par, data, var_v):
    """全試料のカルマンフィルタの負の対数尤度。par = th_inf, th0, log_tau, log_q"""
    th_inf, th0, log_tau, log_q = par
    tau, q = np.exp(log_tau), np.exp(log_q)
    nll = 0.0
    for t, y, n in data:
        m, P = th0, 25.0                    # 最初の状態：平均 th0、分散 25（標準偏差 5°）の事前の幅
        t_prev = 0.0
        for tk, yk, nk in zip(t, y, n):
            phi = np.exp(-(tk - t_prev) / tau)
            m = th_inf + (m - th_inf) * phi
            P = phi**2 * P + q * (1 - phi**2)     # OU 過程の、時間間隔に応じた揺らぎ
            S = P + var_v / nk
            e = yk - m
            nll += 0.5 * (np.log(2 * np.pi * S) + e * e / S)
            K = P / S
            m, P = m + K * e, (1 - K) * P
            t_prev = tk
    return nll


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("angles", nargs="?")
    ap.add_argument("--demo", default=None)
    ap.add_argument("--col", default="theta_tan")
    ap.add_argument("--cut", type=float, default=None, help="この時刻（分）までのデータだけで当てはめた結果も出す")
    args = ap.parse_args()
    if args.demo:
        make_demo(args.demo)
        args.angles = args.demo
    if not args.angles:
        sys.exit("CSV を指定するか、--demo を付ける")
    raw, g = load(args.angles, args.col)
    specs = sorted(g.spec.unique())
    print(f"試料 {len(specs)} 個：{specs}、時刻の数 {g.groupby('spec').size().to_dict()}")

    # 滴どうしの分散（同じ試料・同じ時刻の滴から。プールした分散）
    ok = g["count"] >= 2
    var_v = float(np.sum(g.loc[ok, "var"] * (g.loc[ok, "count"] - 1)) / np.sum(g.loc[ok, "count"] - 1))
    print(f"滴どうしの標準偏差（プール）：{np.sqrt(var_v):.2f}°\n")

    # 1) 試料ごとの非線形回帰（時刻ごとの平均に当てはめる）
    print("=== 非線形回帰（試料ごと）  theta(t) = th_inf − (th_inf − th0)·exp(−t/tau)")
    res = {}
    for s in specs:
        d = g[g.spec == s].sort_values("t")
        p, cov = fit_nls(d.t.values.astype(float), d["mean"].values)
        se = np.sqrt(np.diag(cov))
        dfree = len(d) - 3
        q = stats.t.ppf(0.975, dfree)
        res[s] = p
        print(f"  {s}: th0 {p[1]:.1f}°、th_inf {p[0]:.1f}° [{p[0]-q*se[0]:.1f}, {p[0]+q*se[0]:.1f}]、"
              f"tau {np.exp(p[2]):.0f} 分 [{np.exp(p[2]-q*se[2]):.0f}, {np.exp(p[2]+q*se[2]):.0f}]（自由度 {dfree}）")
    lt = np.array([res[s][2] for s in specs])
    if len(specs) >= 2:
        h = stats.t.ppf(0.975, len(specs) - 1) * lt.std(ddof=1) / np.sqrt(len(specs))
        print(f"  試料を単位にした tau の幾何平均 {np.exp(lt.mean()):.0f} 分、95%区間 [{np.exp(lt.mean()-h):.0f}, "
              f"{np.exp(lt.mean()+h):.0f}]（試料 {len(specs)} 個、自由度 {len(specs)-1}。少ないと広い）")
    # 【誤り】の例：全試料のすべての滴を独立とみなしてまとめて当てはめる
    p_all, cov_all = fit_nls(raw.t.values.astype(float), raw.theta.values)
    se = np.sqrt(np.diag(cov_all))
    q = stats.t.ppf(0.975, len(raw) - 3)
    print(f"  【誤り】全部の滴を独立とみなした当てはめ：tau {np.exp(p_all[2]):.0f} 分 "
          f"[{np.exp(p_all[2]-q*se[2]):.0f}, {np.exp(p_all[2]+q*se[2]):.0f}]（n={len(raw)}。試料の違いを無視して区間が狭すぎる）")

    # 2) 状態空間モデル（全試料で共通のパラメータ）
    data = [(g[g.spec == s].sort_values("t").t.values.astype(float),
             g[g.spec == s].sort_values("t")["mean"].values,
             g[g.spec == s].sort_values("t")["count"].values) for s in specs]
    x0 = [np.mean([r[0] for r in res.values()]), np.mean([r[1] for r in res.values()]), lt.mean(), np.log(4.0)]
    opt = minimize(kalman_nll, x0, args=(data, var_v), method="Nelder-Mead",
                   options={"maxiter": 4000, "xatol": 1e-4, "fatol": 1e-6})
    th_inf, th0, log_tau, log_q = opt.x
    # 標準誤差：数値的なヘッセ行列から
    eps = np.array([0.05, 0.05, 0.01, 0.02])
    Hm = np.zeros((4, 4))
    for i in range(4):
        for j in range(4):
            ei, ej = np.eye(4)[i] * eps[i], np.eye(4)[j] * eps[j]
            Hm[i, j] = (kalman_nll(opt.x + ei + ej, data, var_v) - kalman_nll(opt.x + ei - ej, data, var_v)
                        - kalman_nll(opt.x - ei + ej, data, var_v) + kalman_nll(opt.x - ei - ej, data, var_v)) / (4 * eps[i] * eps[j])
    try:
        se = np.sqrt(np.diag(np.linalg.inv(Hm)))
    except np.linalg.LinAlgError:
        se = np.full(4, np.nan)
    print("\n=== 状態空間モデル（OU 過程＋観測雑音、カルマンフィルタで最尤推定。全試料で共通）")
    print(f"  th0 {th0:.1f}°、th_inf {th_inf:.1f}° ± {1.96*se[0]:.1f}、tau {np.exp(log_tau):.0f} 分 "
          f"[{np.exp(log_tau-1.96*se[2]):.0f}, {np.exp(log_tau+1.96*se[2]):.0f}]、揺らぎの標準偏差 {np.exp(log_q/2):.2f}°"
          f"（収束 {opt.success}、−対数尤度 {opt.fun:.1f}）")
    print("  （揺らぎの戻りの時定数も tau と同じと仮定している。試料ごとの違いは揺らぎ w に吸収される。")
    print("    試料ごとの th_inf の違いが大きいときは階層モデルにする：発展課題）")

    # 3) 外挿の危うさ：短い時間のデータだけで当てはめる
    cut_rows = []
    if args.cut:
        print(f"\n=== {args.cut:g} 分までのデータだけで当てはめると（外挿）")
        for s in specs:
            d = g[(g.spec == s) & (g.t <= args.cut)].sort_values("t")
            if len(d) < 4:
                print(f"  {s}: 点が {len(d)} 個しかなく当てはめられない")
                continue
            try:
                p, cov = fit_nls(d.t.values.astype(float), d["mean"].values)
                se = np.sqrt(np.diag(cov))
                r = cov[0, 2] / np.sqrt(cov[0, 0] * cov[2, 2])
                print(f"  {s}: th_inf {p[0]:.1f}° ± {1.96*se[0]:.1f}、tau {np.exp(p[2]):.0f} 分、"
                      f"th_inf と log tau の推定値の相関 {r:+.2f}（全データでは th_inf {res[s][0]:.1f}°）")
                cut_rows.append((s, p))
            except RuntimeError:
                print(f"  {s}: 当てはめが収束しない（th_inf と tau が区別できない＝識別できない）")
        print("  → 観測した時間が tau より短いと、th_inf と tau を同時には決められない（片方を大きくしても、もう片方で補える）。")

    # 図
    fig, ax = plt.subplots(figsize=(8, 4.8))
    tt = np.geomspace(1, g.t.max() * 1.5, 300)
    for k, s in enumerate(specs):
        d = g[g.spec == s]
        col = f"C{k}"
        ax.errorbar(d.t, d["mean"], yerr=np.sqrt(var_v / d["count"]) * 1.96, fmt="o", color=col, ms=4,
                    capsize=2, label=f"{s}（時刻ごとの平均）")
        ax.plot(tt, model(tt, *res[s]), color=col, lw=1.2)
    for s, p in cut_rows:
        ax.plot(tt, model(tt, *p), "--", color="gray", lw=1)
    ax.plot(tt, model(tt, th_inf, th0, log_tau), color="k", lw=2, label="状態空間モデル（共通の平均の曲線）")
    ax.set_ylim(max(0, g["mean"].min() - 10), min(180, g["mean"].max() + 20))   # 外挿の曲線で縦軸が伸びないように
    if args.cut:
        ax.axvline(args.cut, color="gray", ls=":", lw=1)
        ax.text(args.cut * 1.1, ax.get_ylim()[0] + 1, f"{args.cut:g}分までで当てはめ（破線）", fontsize=8)
    ax.set_xscale("log")
    ax.set_xlabel("処理（洗浄）からの時間 [分]（対数目盛）")
    ax.set_ylabel("接触角 [°]")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("recovery.png", dpi=130)
    print("\n図 → recovery.png")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bo_core.py
#!/usr/bin/env python3
"""第8章：ガウス過程回帰と獲得関数（EI）の共通部分（bo_sim.py・bo_next.py から読み込む）

約束：入力 x は各軸を 0〜1 に直したもの。目的は「最小化」（大きくしたいときは符号を反転して渡す）。
"""
import warnings

import numpy as np
from scipy.stats import norm
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


def fit_gp(X, y, noise_sd=None, ls_bounds=(0.1, 2.0), seed=0):
    """ガウス過程を当てはめる。y は平均を引いて扱う。
    noise_sd を与えたとき：観測雑音の分散を固定（同じ条件のくり返しから求めた値を使う）
    noise_sd が None のとき：WhiteKernel で雑音の大きさも周辺尤度の最大化で推定する（点が少ないと不安定）
    ls_bounds：長さスケールを探す範囲。下限 0.1 は「応答は範囲の1/10 より細かくは波打たない」という事前の知識"""
    X, y = np.asarray(X, float), np.asarray(y, float)
    sd = max(np.std(y), 1e-6)
    k = ConstantKernel(sd**2, (1e-2 * sd**2, 1e2 * sd**2)) * \
        Matern(length_scale=[0.3] * X.shape[1], length_scale_bounds=ls_bounds, nu=2.5)
    if noise_sd is None:
        k = k + WhiteKernel((0.1 * sd) ** 2, (1e-6 * sd**2, sd**2))
        alpha = 1e-8
    else:
        alpha = noise_sd**2
    gp = GaussianProcessRegressor(kernel=k, alpha=alpha, normalize_y=False,
                                  n_restarts_optimizer=5, random_state=seed)
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always", ConvergenceWarning)
        gp.fit(X, y - y.mean())
    # 長さスケールなどが探す範囲の端に張り付いたら、sklearn は ConvergenceWarning を出す。記録しておく
    gp.at_bound_ = any(issubclass(x.category, ConvergenceWarning) for x in w)
    gp.y_mean_ = y.mean()
    return gp


def predict(gp, X, latent=True):
    """事後平均と標準偏差。latent=True なら「真の値 f(x)」の不確かさ（WhiteKernel の雑音分を除く）"""
    mu, sd = gp.predict(np.asarray(X, float), return_std=True)
    if latent and hasattr(gp.kernel_, "k2") and isinstance(gp.kernel_.k2, WhiteKernel):
        sd = np.sqrt(np.maximum(sd**2 - gp.kernel_.k2.noise_level, 1e-12))
    return mu + gp.y_mean_, sd


def expected_improvement(mu, sd, best, xi=0.0):
    """最小化の EI：E[max(best − f − xi, 0)]。best は「今までで一番よいと推定される値」"""
    sd = np.maximum(sd, 1e-12)
    z = (best - mu - xi) / sd
    return (best - mu - xi) * norm.cdf(z) + sd * norm.pdf(z)


def gp_posterior_numpy(gp, X, Xs):
    """式どおりの事後平均・分散（sklearn の結果と一致するかを確かめる用）
       mu* = k*^T (K + σ_n^2 I)^-1 y,  var* = k** − k*^T (K + σ_n^2 I)^-1 k*"""
    k = gp.kernel_
    y = gp.y_train_
    K = k(X) + gp.alpha * np.eye(len(X))
    L = np.linalg.cholesky(K)
    a = np.linalg.solve(L.T, np.linalg.solve(L, y))
    Ks = k(Xs, X)
    mu = Ks @ a
    v = np.linalg.solve(L, Ks.T)
    var = k.diag(Xs) - np.sum(v * v, axis=0)
    return mu + gp.y_mean_, np.sqrt(np.maximum(var, 0))


def grid(n=101):
    g = np.linspace(0, 1, n)
    G1, G2 = np.meshgrid(g, g)
    return np.c_[G1.ravel(), G2.ravel()], G1, G2


def true_surface(X):
    """練習用の『真の応答曲面』（合成）。x1＝処理時間、x2＝距離（どちらも 0〜1 に直したもの）、値は接触角 [°]。
    いちばん下がるのは (0.65, 0.30) 付近で約46.5°。(0.2, 0.8) 付近に浅い谷（局所的な最小）がある"""
    X = np.atleast_2d(X)
    x1, x2 = X[:, 0], X[:, 1]
    g1 = np.exp(-((x1 - 0.65) ** 2 / 0.06 + (x2 - 0.30) ** 2 / 0.03))
    g2 = np.exp(-((x1 - 0.20) ** 2 / 0.02 + (x2 - 0.80) ** 2 / 0.02))
    return 100 - 55 * g1 - 15 * g2 + 5 * x2

In [ ]:
%%writefile bo_sim.py
#!/usr/bin/env python3
"""第8章：合成の「真の応答曲面」でベイズ最適化を練習し、ランダム探索・格子と比べる

使い方: cd ~/home-lab/ch8 && uv run python bo_sim.py --budget 15 --init 5 --reps 30 --noise 2
        uv run python bo_sim.py --show 1        （1回分の様子を図にする：bo_sim.png）

設定：接触角を下げたい（最小化）。1回の「実験」で、真の値に標準偏差 --noise の雑音が乗った値が返る
比べる方法（どれも同じ回数 --budget だけ実験する）
  BO      ：最初の --init 点はランダム、そのあとは GP＋EI で次の点を選ぶ
  random  ：全部ランダム
  grid    ：budget 以下で最大の、正方形に近い格子
推薦する条件：最後に GP を当てはめ、実験した点のうち事後平均が最小の点（どの方法も同じ規則）
報告する値
  regret       ：推薦した条件の『真の値』 − 真の最小値（小さいほどよい）
  【誤り】best_y：実験で観測した値の最小（雑音で運よく低く出た点を選ぶので、真の値より楽観的になる）
"""
import argparse

import numpy as np
from scipy.stats import wilcoxon
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import bo_core as bc

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
XG, G1, G2 = bc.grid(101)
F_TRUE = bc.true_surface(XG)
F_MIN = F_TRUE.min()


def experiment(X, rng, noise):
    return bc.true_surface(X) + rng.normal(0, noise, len(X))


def recommend(X, y, noise):
    gp = bc.fit_gp(X, y, noise_sd=noise)
    mu, _ = bc.predict(gp, X)
    return X[np.argmin(mu)], gp


def run_bo(rng, budget, n_init, noise, known_noise=True, log=False, xi=0.0):
    X = rng.random((n_init, 2))
    y = experiment(X, rng, noise)
    while len(X) < budget:
        gp = bc.fit_gp(X, y, noise_sd=noise if known_noise else None)
        mu_obs, _ = bc.predict(gp, X)
        best = mu_obs.min()                           # 雑音があるので、観測値の最小ではなく事後平均の最小を基準にする
        mu, sd = bc.predict(gp, XG)
        ei = bc.expected_improvement(mu, sd, best, xi)
        xn = XG[np.argmax(ei)][None, :]
        if log:
            print(f"  n={len(X):2d}  カーネル {gp.kernel_}" + ("  ← 探す範囲の端に張り付いた" if gp.at_bound_ else ""))
        X = np.vstack([X, xn])
        y = np.r_[y, experiment(xn, rng, noise)]
    return X, y


def run_random(rng, budget, noise):
    X = rng.random((budget, 2))
    return X, experiment(X, rng, noise)


def run_grid(rng, budget, noise):
    a = int(np.floor(np.sqrt(budget)))
    b = budget // a
    g1, g2 = np.meshgrid((np.arange(a) + 0.5) / a, (np.arange(b) + 0.5) / b)
    X = np.c_[g1.ravel(), g2.ravel()]
    return X, experiment(X, rng, noise)


def show(seed, budget, n_init, noise):
    rng = np.random.default_rng(seed)
    print("1回分の BO（カーネルのハイパーパラメータが点の数でどう変わるか）")
    X, y = run_bo(rng, budget, n_init, noise, log=True)
    xr, gp = recommend(X, y, noise)
    mu, sd = bc.predict(gp, XG)
    # 自前の式と sklearn の一致を確かめる
    mu2, sd2 = bc.gp_posterior_numpy(gp, X, XG)
    print(f"自前の式と sklearn の差：平均 最大 {np.max(np.abs(mu - mu2)):.2e}、標準偏差 最大 {np.max(np.abs(sd - sd2)):.2e}")
    best = bc.predict(gp, X)[0].min()
    ei = bc.expected_improvement(mu, sd, best)
    fig, ax = plt.subplots(1, 4, figsize=(15, 3.8))
    for a, Z, t in zip(ax, (F_TRUE, mu, sd, ei),
                       ("真の応答曲面（合成）", f"GP の事後平均（{len(X)} 点）", "事後標準偏差", "次の EI")):
        c = a.contourf(G1, G2, Z.reshape(G1.shape), 20, cmap="viridis")
        fig.colorbar(c, ax=a, shrink=0.8)
        a.plot(X[:n_init, 0], X[:n_init, 1], "wo", ms=5, mec="k")
        a.plot(X[n_init:, 0], X[n_init:, 1], "w^", ms=6, mec="k")
        a.plot(*xr, "r*", ms=14, mec="k")
        a.set_title(t, fontsize=10)
        a.set_xlabel("x1（処理時間）")
    ax[0].set_ylabel("x2（距離）")
    fig.tight_layout()
    fig.savefig("bo_sim.png", dpi=120)
    print(f"推薦 x = ({xr[0]:.2f}, {xr[1]:.2f})、真の値 {bc.true_surface(xr)[0]:.1f}°（真の最小 {F_MIN:.1f}°）"
          f"、観測値の最小 {y.min():.1f}°  → 図 bo_sim.png")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--budget", type=int, default=15)
    ap.add_argument("--init", type=int, default=5)
    ap.add_argument("--reps", type=int, default=30)
    ap.add_argument("--noise", type=float, default=2.0)
    ap.add_argument("--xi", type=float, default=0.0, help="EI の xi（大きくすると探索寄り）[°]")
    ap.add_argument("--show", type=int, default=None, help="この乱数の種で1回だけ行い、図を描く")
    args = ap.parse_args()
    if args.show is not None:
        show(args.show, args.budget, args.init, args.noise)
        return
    print(f"真の最小値 {F_MIN:.1f}°、実験 {args.budget} 回（BO は最初の {args.init} 回がランダム）、"
          f"雑音 {args.noise}°、{args.reps} 回くり返し")
    res = {m: {"regret": [], "best_y": [], "true_at_best_y": []} for m in ("BO", "BO（雑音も推定）", "random", "grid")}
    for r in range(args.reps):
        for m in res:
            rng = np.random.default_rng(1000 + r)          # 同じ r では同じ乱数から始める（対応のある比較）
            if m == "BO":
                X, y = run_bo(rng, args.budget, args.init, args.noise, xi=args.xi)
            elif m == "BO（雑音も推定）":
                X, y = run_bo(rng, args.budget, args.init, args.noise, known_noise=False, xi=args.xi)
            elif m == "random":
                X, y = run_random(rng, args.budget, args.noise)
            else:
                X, y = run_grid(rng, args.budget, args.noise)
            xr, _ = recommend(X, y, args.noise)
            res[m]["regret"].append(bc.true_surface(xr)[0] - F_MIN)
            res[m]["best_y"].append(y.min())
            res[m]["true_at_best_y"].append(bc.true_surface(X[np.argmin(y)])[0])
        if (r + 1) % 10 == 0:
            print(f"  {r+1}/{args.reps}")
    print(f"\n方法 / regret 中央値 [四分位] / 失敗（regret > 10°）の回数 / 【誤り】観測の最小の平均 / その点の真の値の平均")
    for m, d in res.items():
        q = np.percentile(d["regret"], [25, 50, 75])
        nf = int(np.sum(np.array(d["regret"]) > 10))
        print(f"  {m:<12} {q[1]:5.1f} [{q[0]:4.1f}, {q[2]:4.1f}]   {nf:2d}/{args.reps}   "
              f"{np.mean(d['best_y']):6.1f}°   {np.mean(d['true_at_best_y']):6.1f}°")
    opt = np.mean(np.array(res["BO"]["true_at_best_y"]) - np.array(res["BO"]["best_y"]))
    print(f"\n観測の最小値は、その点の真の値より平均 {opt:.1f}° 低い（楽観的）。")
    print("→ 『いちばんよかった実験の値』をそのまま性能として報告しない。推薦した条件で、改めて何回か測り直して報告する。")
    for m in ("random", "grid"):
        d = np.array(res[m]["regret"]) - np.array(res["BO"]["regret"])
        p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        print(f"BO と {m} の regret の差（{m} − BO）：中央値 {np.median(d):+.1f}°、ウィルコクソン p = {p:.3g}"
              f"（{args.reps} 回の対応のある比較）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bo_next.py
#!/usr/bin/env python3
"""第8章：実物の実験で、次に試す研磨条件をベイズ最適化で選ぶ（PP の接触角をいちばん大きくする条件を探す）

使い方: cd ~/home-lab/ch8
  uv run python bo_next.py bo_runs.csv                 （これまでの結果から、次の条件を1つ提案する）
  uv run python bo_next.py --demo bo_runs_demo.csv     （合成の結果で練習する）

bo_runs.csv の列（1行＝1回の実験＝1枚の試料片。滴の値は試料片ごとに平均してから書く）
  run,grit,strokes,theta_mean,n_drops,theta_sd
  grit：紙やすりの番手（#120〜#2000 の、手元にあるもの）、strokes：一方向に何往復こするか（0〜40）
  theta_mean：その試料片の滴の接触角の平均、n_drops：滴の数、theta_sd：滴どうしの標準偏差
入力の直し方：x1 = (log10(grit) − log10(120)) / (log10(2000) − log10(120))、x2 = strokes / 40
観測雑音（1回の実験＝1枚の試料片の平均の、真の値からのばらつき）の決め方
  標準：--noise-sd に、同じ条件で作った2枚の試料片の平均の差 d から |d|/√2 を渡す（何組かあれば2乗平均）。
        この値には、試料片どうしの違いと滴の平均の誤差の両方が入っている
  代わり：--piece-sd に試料片どうしの標準偏差 σ_u（doe_anova.py の線形混合モデルの「試料片」の分散成分の平方根）を渡すと、
        √(平均(theta_sd²/n_drops) + σ_u²) を使う。分散分析の残差の標準偏差は、滴の平均の誤差をすでに含むので
        --piece-sd に渡さない（二重に数えることになる）
"""
import argparse
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import bo_core as bc

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
G_LO, G_HI, S_MAX = np.log10(120), np.log10(2000), 40.0
INIT = [(240, 10), (1000, 30), (400, 20), (120, 35), (2000, 5)]   # 最初の5回（範囲に散らばるように選んだ）


def to_x(grit, strokes):
    return np.c_[(np.log10(grit) - G_LO) / (G_HI - G_LO), np.asarray(strokes, float) / S_MAX]


def from_x(x):
    return 10 ** (G_LO + x[..., 0] * (G_HI - G_LO)), x[..., 1] * S_MAX


def make_demo(path, seed=3):
    """合成の結果：真の曲面は「細かすぎず粗すぎない番手を、ある程度の回数」で接触角が最大（約112°）"""
    rng = np.random.default_rng(seed)
    rows = []
    for i, (g, s) in enumerate(INIT, start=1):
        x = to_x(g, s)[0]
        f = 100 + 12 * np.exp(-((x[0] - 0.45) ** 2 / 0.08 + (x[1] - 0.6) ** 2 / 0.15)) - 4 * x[1] * (x[0] < 0.15)
        vals = f + rng.normal(0, 1.5) + rng.normal(0, 2.0, 3)
        rows.append({"run": i, "grit": g, "strokes": s, "theta_mean": vals.mean().round(2),
                     "n_drops": 3, "theta_sd": vals.std(ddof=1).round(2)})
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"合成の結果 → {path}（最初の5回分）\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="?")
    ap.add_argument("--demo", default=None)
    ap.add_argument("--grits", type=int, nargs="+", default=[120, 240, 400, 600, 800, 1000, 1500, 2000],
                    help="手元にある紙やすりの番手")
    ap.add_argument("--noise-sd", type=float, default=None, help="1回の実験の雑音の標準偏差 [°]（同じ条件の2枚の差 d から |d|/√2）")
    ap.add_argument("--piece-sd", type=float, default=1.5, help="試料片どうしの標準偏差 σ_u [°]（混合モデルの分散成分。--noise-sd がないとき）")
    ap.add_argument("--xi", type=float, default=0.0)
    args = ap.parse_args()
    if args.demo:
        make_demo(args.demo)
        args.runs = args.demo
    if not args.runs or not pathlib.Path(args.runs).exists():
        print("まだ結果がありません。最初の5回はこの条件で行う（順番は乱数で入れ替える）：")
        for g, s in INIT:
            print(f"  #{g}・{s}往復")
        sys.exit(0)
    d = pd.read_csv(args.runs)
    if len(d) < len(INIT):
        print(f"結果が {len(d)} 回分です。GP を当てはめる前に、最初の5回の残りを行う：")
        for g, s in INIT[len(d):]:
            print(f"  #{g}・{s}往復")
        sys.exit(0)
    X = to_x(d.grit.values, d.strokes.values)
    y = -d.theta_mean.values                         # 大きくしたいので符号を反転して「最小化」
    if args.noise_sd is not None:
        se = args.noise_sd
        print(f"{len(d)} 回分の結果。観測雑音の標準偏差（--noise-sd）= {se:.2f}°")
    else:
        se = np.sqrt(np.mean(d.theta_sd.values ** 2 / d.n_drops.values) + args.piece_sd ** 2)
        print(f"{len(d)} 回分の結果。観測雑音の標準偏差（滴の平均の標準誤差＋試料片のばらつき σ_u）≈ {se:.2f}°")
    gp = bc.fit_gp(X, y, noise_sd=se)
    print(f"当てはめたカーネル：{gp.kernel_}" + ("（範囲の端に張り付いた：点が少ないうちはよくある。長さスケールの値を信じすぎない）" if gp.at_bound_ else ""))
    mu_obs, sd_obs = bc.predict(gp, X)
    best = mu_obs.min()
    print("\nこれまでの条件と、GP による『真の値』の推定（事後平均 ± 標準偏差）")
    for (_, r), m, s in zip(d.iterrows(), mu_obs, sd_obs):
        print(f"  run {int(r.run):2d}  #{int(r.grit):<5d} {int(r.strokes):2d}往復  観測 {r.theta_mean:6.1f}°   推定 {-m:6.1f} ± {s:.1f}°")
    # 候補：手元の番手 × 0〜40往復（2往復きざみ）
    gg, ss = np.meshgrid(args.grits, np.arange(0, 41, 2))
    C = to_x(gg.ravel(), ss.ravel())
    mu, sd = bc.predict(gp, C)
    ei = bc.expected_improvement(mu, sd, best, args.xi)
    order = np.argsort(-ei)
    print("\n次に試す候補（EI の大きい順。上から1つを選び、結果を bo_runs.csv に1行足してもう一度実行）")
    for i in order[:5]:
        print(f"  #{gg.ravel()[i]:<5d} {ss.ravel()[i]:2d}往復   予測 {-mu[i]:6.1f} ± {sd[i]:.1f}°   EI {ei[i]:.3f}")
    ib = int(np.argmin(mu))
    print(f"\n今の推薦（事後平均が最大）：#{gg.ravel()[ib]}・{ss.ravel()[ib]}往復、予測 {-mu[ib]:.1f} ± {sd[ib]:.1f}°")
    print("報告するときは、推薦の条件で新しい試料片を2枚以上作って測り直した値を使う（探索に使った値で性能を言わない）。")

    # 図：事後平均と標準偏差（横軸は番手の対数）
    xg, G1, G2 = bc.grid(81)
    m2, s2 = bc.predict(gp, xg)
    gr, st = from_x(xg)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a, Z, t in zip(ax, (-m2, s2), ("事後平均 [°]", "事後標準偏差 [°]")):
        c = a.contourf(gr.reshape(G1.shape), st.reshape(G1.shape), Z.reshape(G1.shape), 20, cmap="viridis")
        fig.colorbar(c, ax=a)
        a.plot(d.grit, d.strokes, "wo", mec="k")
        a.plot(gg.ravel()[order[0]], ss.ravel()[order[0]], "r*", ms=15, mec="k", label="次の候補")
        a.set_xscale("log")
        a.set_xlabel("紙やすりの番手（対数目盛）")
        a.set_ylabel("往復の回数")
        a.set_title(t)
    ax[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("bo_next.png", dpi=120)
    print("図 → bo_next.png")


if __name__ == "__main__":
    main()

In [ ]:
import glob, os, re, shlex, subprocess, sys
from IPython.display import Image, display


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    words = []
    for w in shlex.split(args):   # synth/cap_*_refl.png のような * を、ターミナルと同じく名前の順に広げる
        words += (sorted(glob.glob(w)) or [w]) if any(c in w for c in "*?[") else [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines() if "findfont" not in l)   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
    if err.strip():
        print(err, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=640):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


ANG = re.compile(r"^(\S+?)\.png: 接線 ([\d.]+)°（左右の差 ([-+\d.]+)°） 楕円 ([\d.]+)°  円 ([\d.]+)°  θ/2 ([\d.]+)°"
                 r"  傾き ([-+\d.]+)°(?:  体積 ([\d.]+) µL  Bo ([\d.]+))?", re.M)


def angles(text):
    """contact_angle.py の表示を {名前: (接線, 左右の差, 楕円, 円, θ/2, 傾き, 体積, Bo)} にする"""
    return {m.group(1): tuple(float(v) if v else None for v in m.groups()[1:]) for m in ANG.finditer(text)}


for f in ["synth_angles.csv", "design_ch8.csv"]:   # やり直したとき：CSV は追記になり、実験表は上書きしないので、先に消す
    if os.path.exists(f):
        os.remove(f)
print("準備ができました")

## 1. 合成の滴の画像で、接触角の計算を確かめる（8-3-3、手順 A の 2）

`run_ch8.sh` の A の6行です（数十秒）。`drop_synth.py` は、答えのわかった影絵（傾き 1.5°・ぼけ・雑音・真ん中の明るい点つき）を `synth/` に作ります。`contact_angle.py` は接触角を4通り（接線・楕円・円・θ/2）で求め、確認の図を `check/` に保存します。図で、オレンジの点（輪郭）が影のふちに乗り、水色の線（基線）が基板の上面に重なっていることを見てください。

まず球冠（重力なし、接触半径 160画素）の 30°・70°・90°・95°・100°・110° です。

In [ ]:
run("drop_synth.py --theta 30 70 90 95 100 110 --out synth")
out_cap = run("contact_angle.py synth/cap_30.png synth/cap_70.png synth/cap_90.png synth/cap_95.png synth/cap_100.png synth/cap_110.png --csv synth_angles.csv")

In [ ]:
# 確かめ（手順 A の 2 と 8-7）
a = angles(out_cap)
true = {"cap_30": 30, "cap_70": 70, "cap_90": 90, "cap_95": 95, "cap_100": 100, "cap_110": 110}
ok(set(a) == set(true), "6枚とも測れた")
ok(all(abs(a[k][0] - t) <= 0.5 for k, t in true.items()), "接線法が ±0.5° 以内 " + str([a[k][0] for k in true]))
ok(all(abs(a[k][j] - t) <= 0.1 + 1e-9 for k, t in true.items() for j in (2, 3, 4) if t in (30, 70, 110)),
   "30°・70°・110° で楕円・円・θ/2 法が ±0.1° 以内")
ok(all(abs(a[k][5] + 1.50) < 0.005 for k in true), "傾き −1.50°")
near = lambda got, want, tol=0.15: len(got) == len(want) and all(abs(g - w) <= tol + 1e-9 for g, w in zip(got, want))   # 版で最後の桁が変わる分の幅
ok(near([a[k][0] for k in true], [30.0, 70.3, 89.9, 95.0, 100.2, 110.2]) and near([a[k][3] for k in true], [30.0, 70.0, 90.0, 95.0, 100.0, 110.0]),
   "8-7 の値と ±0.15° 以内で同じ：接線法 30.0・70.3・89.9・95.0・100.2・110.2°、円の当てはめ 30.0・70.0・90.0・95.0・100.0・110.0° "
   + str([a[k][0] for k in true]) + " " + str([a[k][3] for k in true]))
show("check/cap_30_check.png", "check/cap_110_check.png")

次に、Young–Laplace の式で作った 5µL の滴（重力で少しつぶれる）です。円の当てはめと θ/2 法は、重力のせいで小さく出ます（図8-2）。

In [ ]:
run("drop_synth.py --theta 30 70 110 --model yl --volume-ul 5 --out synth")
out_yl = run("contact_angle.py synth/yl_30.png synth/yl_70.png synth/yl_110.png --csv synth_angles.csv --mm-per-px 0.008")

In [ ]:
# 確かめ（手順 A の 2 と 8-7）
a = angles(out_yl)
true = {"yl_30": 30, "yl_70": 70, "yl_110": 110}
ok(all(abs(a[k][0] - t) <= 0.3 for k, t in true.items()), "接線法が ±0.3° 以内 " + str([a[k][0] for k in true]))
ok(near([a[k][3] for k in true], [28.7, 67.7, 105.3]), "円の当てはめは ±0.15° 以内で 28.7°・67.7°・105.3°（重力で小さく出る）" + str([a[k][3] for k in true]))
ok(all(a[k][3] < t and a[k][4] < t for k, t in true.items()), "円と θ/2 法はどれも答えより小さい " + str([a[k][4] for k in true]))
ok(all(4.98 <= a[k][6] <= 5.01 for k in true), "体積 5.00µL（4.99〜5.00）" + str([a[k][6] for k in true]))
show("check/yl_110_check.png")

最後に、基板への映り込みのある画像（傾き 0°）を `--baseline mirror`（水滴と映り込みの折れ曲がりから基線を決める）で測ります。90°・95°・100° は、水滴と映り込みの輪郭がまっすぐつながって折れ曲がりがないので、「基線が決まりません」と表示されて**とばされるのが正しい動き**です（`run` は、とばされても止まりません）。

In [ ]:
run("drop_synth.py --theta 30 70 90 95 100 110 --reflect --tilt 0 --out synth")
out_refl = run("contact_angle.py synth/cap_*_refl.png --baseline mirror --csv synth_angles.csv")

In [ ]:
# 確かめ（手順 A の 2 と 8-7）
a = angles(out_refl)
true = {"cap_30_refl": 30, "cap_70_refl": 70, "cap_110_refl": 110}
ok(set(a) == set(true) and all(abs(a[k][0] - t) <= 0.5 for k, t in true.items()),
   "30°・70°・110° だけが測れて、接線法で ±0.5° 以内 " + str([a[k][0] for k in true]))
skipped = re.findall(r"とばした：synth/(cap_\d+)_refl\.png: --baseline mirror では θ が 90° 付近", out_refl)
ok(sorted(skipped) == ["cap_100", "cap_90", "cap_95"], "90°・95°・100° は「基線が決まりません」でとばされた")
show("check/cap_70_refl_check.png")

## 2. スケール（mm/画素）を求める（8-3-3、手順 B の 4 を合成画像で）

教科書では、1円玉（直径 20.0mm）をつまようじに立てて撮った写真 `photos/coin.jpg` で `contact_angle.py --calib` を実行します。ここでは `drop_synth.py --coin` の合成の影絵（0.025 mm/画素、直径 800画素、上が画面からはみ出す）で、同じ計算を確かめます。実物の写真では mm/画素 が 0.006〜0.0075 程度になります（12M画素で横 25〜30mm）。

In [ ]:
run("drop_synth.py --coin --mm-per-px 0.025 --out synth")
out_coin = run("contact_angle.py --calib synth/coin.png --calib-mm 20.0")

In [ ]:
# 確かめ（手順 B の 4：RMS が1画素以下、見えている円弧が 120° 以上、直径が答えに近い）
D = num(r"円板の直径 ([\d.]+) 画素", out_coin)
ok(abs(D - 800) <= 0.5, f"直径 {D} 画素（答え 800。教科書の合成画像では 799.9）")
ok(num(r"円からのずれ RMS ([\d.]+) 画素", out_coin) <= 1 and num(r"見えている円弧 ([\d.]+)°", out_coin) >= 120,
   "円からのずれ RMS が1画素以下、見えている円弧が 120° 以上")
ok(abs(num(r"→ ([\d.]+) mm/画素", out_coin) - 0.025) < 2e-5, "0.025 mm/画素 に戻った")

## 3. 2<sup>3</sup> 要因計画（8-3-4・8-3-5、手順 D の 7・11）

`design_ch8.py` で直交表 L8 と、16回分（2ブロック × 8条件、各3滴）の無作為化した実験表を作ります。教科書の `run_ch8.sh` は `--seed` を付けず（毎回ちがう種を使い、表示された種をノートに書く）、8-7 の値は `--seed 8` の表のものです。ここでは 8-7 と比べられるように **`--seed 8` を足して**います（自分の実験の表を作るときは `--seed` を付けずに Mac で作り、種をノートに書いてください）。

次に `doe_anova.py --demo` で、合成データ（真の効果：A −25°、B 0°、A×B −10°、C −3°、ブロック +2°）を作って解析し、出力の読み方を練習します。「試料片が単位」（正しい。残差の自由度7）と「滴が単位」（疑似反復。自由度39）で、効果の推定値はほぼ同じなのに区間の幅と p が違うことを見てください。

In [ ]:
run("design_ch8.py --blocks 2 --drops 3 --seed 8 --out design_ch8.csv")   # 教科書の run_ch8.sh は --seed なし

In [ ]:
# 確かめ（手順 D の 7）
import pandas as pd
d = pd.read_csv("design_ch8.csv")
ok(len(d) == 16 and all(len(g.drop_duplicates(["material", "sanding", "cleaning"])) == 8 for _, g in d.groupby("block")),
   "16回分で、ブロックごとに8条件が1回ずつ")
ok(all((g["material"] == "PP").sum() == 4 for _, g in d.groupby("block")), "各ブロックで PP と PET が4枚ずつ")
ok(d.piece.nunique() == 16, "試料片 P1〜P8・E1〜E8 が1回ずつ使われる")

In [ ]:
out_doe = run("doe_anova.py design_ch8.csv --demo demo_angles.csv")
show("doe_effects.png")

In [ ]:
# 確かめ（手順 D の 11 と 8-7。--seed 8 の表と合成データの値）
row = lambda name, part: [float(v) for v in re.search(rf"^{re.escape(name)}\s+([-\d.]+)\s+([-\d.]+)\s+([-\d.]+)\s+([\d.]+)\s+([\d.]+)$", part, re.M).groups()]
piece, drop = out_doe.split("【誤り】滴を単位")[0], out_doe.split("【誤り】滴を単位")[1].split("線形混合モデル")[0]
ok("Residual    29.3441  7.0" in piece and "Residual   180.1353  39.0" in drop, "残差の自由度が 7（試料片が単位）と 39（滴が単位）")
A, AB, Bm, C, ABC = (row(k, piece) for k in ("A", "A×B", "B", "C", "A×B×C"))
ok([round(v, 1) for v in A[:3]] == [-26.0, -28.4, -23.6] and [round(v, 1) for v in AB[:3]] == [-9.7, -12.1, -7.3],
   f"A の効果 −26.0°［−28.4, −23.6］、A×B −9.7°［−12.1, −7.3］")
ok([round(v, 1) for v in Bm[:3]] == [1.2, -1.3, 3.6] and round(C[0], 1) == -4.0 and C[4] == 0.029,
   "B の主効果 +1.2°［−1.3, +3.6］、C −4.0°（ホルム補正後 p = 0.029）")
ABCd = row("A×B×C", drop)
ok(ABC[3] == 0.020 and ABC[4] == 0.079 and ABCd[3] == 0.0 and ABCd[4] == 0.0,
   "本当は0 の A×B×C が、試料片が単位では p = 0.020（補正後 0.079）、滴が単位では p < 0.001（疑似反復）")
ok(abs(row("A", drop)[0] - A[0]) < 1e-9 and (row("A", drop)[2] - row("A", drop)[1]) < (A[2] - A[1]),
   "効果の推定値は同じで、滴が単位の区間のほうが狭い")
ok("PET: -8.5°  95%区間 [-15.2, -1.9]" in out_doe and "PP: +10.8°  95%区間 [+6.5, +15.2]" in out_doe,
   "素材ごとの紙やすりの効果は PP で +10.8°［+6.5, +15.2］、PET で −8.5°［−15.2, −1.9］と逆向き")
icc = num(r"級内相関 ICC = ([\d.]+)", out_doe)
ok(abs(icc - 0.5) < 0.1 and "A     -26.00  2.42          2.01" in out_doe,
   f"線形混合モデルの級内相関が約0.5（{icc}）、区間は t（自由度7）で ±2.4°、1.96 なら ±2.0°")

## 4. 疎水性回復（8-3-6、手順 E の 14）

`recovery_fit.py --demo` で、洗ったガラス3枚の合成データ（2分〜2日の11時刻、各3滴）を作り、試料ごとの非線形回帰（$\theta(t) = \theta_\infty - (\theta_\infty - \theta_0)e^{-t/\tau}$）と、状態空間モデル（カルマンフィルタ）で当てはめます。`--cut 120` は、120分までのデータだけで当てはめた結果も出します（外挿の危うさ）。自分のガラスの写真（`G1_0030m_D2.jpg` の形）で行うのは Mac で、です。

In [ ]:
out_rec = run("recovery_fit.py --demo demo_rec.csv --cut 120")
show("recovery.png")

In [ ]:
# 確かめ（手順 E の 14 と 8-7）
taus = [float(v) for v in re.findall(r"^  G\d: th0 [\d.]+°、th_inf [\d.]+° \[[\d.]+, [\d.]+\]、tau (\d+) 分", out_rec, re.M)]
ok(len(taus) == 3 and min(taus) == 347 and max(taus) == 415, f"試料ごとの τ が 347〜415分 {taus}")
ok("幾何平均 372 分、95%区間 [294, 471]" in out_rec and "tau 374 分 [323, 434]" in out_rec,
   "試料を単位にした τ 372分［294, 471］。全部の滴を独立とみなすと［323, 434］と狭くなる")
ok("th_inf 49.0° ± 2.2、tau 357 分 [284, 449]" in out_rec and "収束 True" in out_rec, "状態空間モデル：θ∞ 49.0° ± 2.2°、τ 357分［284, 449］")
cut = [float(v) for v in re.findall(r"^  G\d: th_inf ([\d.]+)° ± [\d.]+、tau", out_rec, re.M)]
full = [float(v) for v in re.findall(r"（全データでは th_inf ([\d.]+)°）", out_rec)]
cor = [float(v) for v in re.findall(r"推定値の相関 ([-+\d.]+)", out_rec)]
ok(all(24 <= c <= 34 for c in cut) and all(47 <= f <= 52 for f in full) and all(0.93 <= r <= 0.97 for r in cor),
   f"120分までだと θ∞ が 24〜34°（{cut}）で、全部のデータの 47〜52°（{full}）を大きく外れる（相関 {cor}）")
ok("G1: th0 14.8°、th_inf 48.5° [46.5, 50.6]" in out_rec, "G1 の区間［46.5, 50.6］は、合成の真の値 θ∞ = 45.4° を含まない（揺らぎを無視した区間は狭すぎる）")

## 5. 合成の曲面でのベイズ最適化（8-3-7、手順 F の 15）

まず `bo_sim.py --show 1` で、1回分の BO（最初の5点はランダム、あとは EI が最大の点）を見ます。図 `bo_sim.png` で、三角（BO が選んだ点）が谷の近くに集まることを見てください。表示では、点が少ないうちはカーネルの長さスケールが「探す範囲の端に張り付く」ことと、自前の式（`bo_core.py`）と scikit-learn の事後平均・標準偏差が丸め誤差の範囲で一致することを見ます。

In [ ]:
out_show = run("bo_sim.py --show 1")
show("bo_sim.png")

In [ ]:
# 確かめ（8-3-7 と手順 F の 15）
stuck = re.findall(r"^  n=\s*(\d+)  カーネル .*?(← 探す範囲の端に張り付いた)?$", out_show, re.M)
ok(len(stuck) >= 6 and all(s for n, s in stuck if int(n) <= 10), "10点までは、長さスケールが探す範囲の端に張り付いた")
ok(num(r"平均 最大 ([\d.e+-]+)", out_show) < 1e-9 and num(r"標準偏差 最大 ([\d.e+-]+)", out_show) < 1e-9,
   "自前の式と sklearn の差が 1e-9 より小さい")
ok(num(r"真の値 ([\d.]+)°（真の最小", out_show) - num(r"真の最小 ([\d.]+)°", out_show) < 2, "推薦した点の真の値が、真の最小値に 2° 以内で近い")

次に、BO・BO（雑音も推定）・ランダム探索・格子を、同じ乱数から何回もくり返して比べます（実験 15 回、最初の5回はランダム、雑音 2°）。**ノートブックの既定は 10 回くり返し**（執筆時の Mac で約6秒）です。教科書の本番（図8-12）は 30 回です。

- regret ＝ 推薦した条件の真の値 − 真の最小値（小さいほどよい）。格子は点の位置が毎回同じなので、毎回 17.2° です。
- 【誤り】の列：観測の最小値は、その点の真の値より低く出ます（楽観的、勝者の呪い）。いちばんよかった実験の値を、そのまま性能として報告しません。
- 30 回のとき、BO と格子の差は中央値 +16.6° と大きいのに、ウィルコクソンの p ≈ 0.06 になります。BO が失敗した回の大きな負の差に大きい順位がつくためです（8-7）。くり返しが 10 回では p はもっと大きく出ます。

In [ ]:
REPS = 10   # 試しの設定。教科書の本番（手順 F の 15・図8-12）は REPS = 30（執筆時の Mac で約16秒、Colab ではその数倍）
out_sim = run(f"bo_sim.py --budget 15 --init 5 --reps {REPS} --noise 2")

In [ ]:
# 確かめ（手順 F の 15。くり返しの数に関係なく成り立つこと。30 回のときは 8-7・図8-12 の値とも比べる）
tab = {m.group(1): (float(m.group(2)), int(m.group(3)), float(m.group(4)), float(m.group(5)))
       for m in re.finditer(r"^  (BO|BO（雑音も推定）|random|grid)\s+([\d.]+) \[\s*[\d.]+,\s*[\d.]+\]\s+(\d+)/\d+\s+([\d.]+)°\s+([\d.]+)°$", out_sim, re.M)}
ok(len(tab) == 4, "4つの方法の表が出た")
ok(tab["grid"][0] == 17.2 and tab["grid"][1] == REPS and f"grid          17.2 [17.2, 17.2]" in out_sim,
   f"格子は毎回 regret 17.2°（失敗 {REPS}/{REPS}）")
ok(tab["BO"][0] < min(tab["random"][0], tab["grid"][0]),
   f"BO の regret の中央値 {tab['BO'][0]}° が、ランダム探索 {tab['random'][0]}°・格子 {tab['grid'][0]}° より小さい")
ok(1 <= tab["BO"][1] < REPS, f"ただし BO にも失敗（regret > 10°）の回がある（{tab['BO'][1]}/{REPS}）")
opt = num(r"観測の最小値は、その点の真の値より平均 ([\d.]+)° 低い", out_sim)
ok(tab["BO"][2] < tab["BO"][3] and 1 < opt < 4, f"観測の最小値が、その点の真の値より平均 {opt}° 低い（楽観的。教科書は 30 回で 2.3°）")
if REPS == 30:
    ok(tab["BO"][:2] == (0.6, 6) and tab["random"][:2] == (15.3, 20), "30 回：BO 0.6°（失敗 6/30）、ランダム探索 15.3°（失敗 20/30）")
    ok(abs(num(r"random − BO）：中央値 \+[\d.]+°、ウィルコクソン p = ([\d.]+)", out_sim) - 0.004) < 0.001
       and num(r"grid − BO）：中央値 \+([\d.]+)°", out_sim) == 16.6
       and abs(num(r"grid − BO）：中央値 \+[\d.]+°、ウィルコクソン p = ([\d.]+)", out_sim) - 0.06) < 0.005
       and opt == 2.3,
       "30 回：ランダム探索との差 p ≈ 0.004、格子との差は中央値 +16.6° なのに p ≈ 0.06、楽観の大きさ 2.3°")

## 6. 研磨条件の次の候補を選ぶ（手順 F の 16・17 を合成の結果で）

教科書では、PP の試料片を実際に紙やすりで研磨して撮り、`bo_runs.csv`（列 `run,grit,strokes,theta_mean,n_drops,theta_sd`）に1行ずつ書いて `bo_next.py` を実行します。ここでは `bo_next.py --demo` が作る合成の最初の5回分（`bo_runs_demo.csv`）で練習します。5点では番手の長さスケールが上限（2）に張り付き、候補が番手の方向に一列に並びます（点が少ないうちの GP は、見えていない方向について何も知らない、ということの例。8-7）。

In [ ]:
out_next = run("bo_next.py --demo bo_runs_demo.csv")
show("bo_next.png")

In [ ]:
# 確かめ（手順 F の 16 と 8-7）
ok(len(re.findall(r"^  run  \d", out_next, re.M)) == 5, "5回分の結果が表示された")
ok("Matern(length_scale=[2," in out_next and "範囲の端に張り付いた" in out_next, "5点では番手の長さスケールが上限 2 に張り付いた")
cand = re.findall(r"^  #(\d+)\s+(\d+)往復   予測", out_next, re.M)
ok(len(cand) == 5 and len({s for g, s in cand}) == 1 and len({g for g, s in cand}) == 5,
   f"候補が、往復の回数は同じで番手だけが違う「番手の方向」に一列に並んだ {cand}")
sd = num(r"試料片のばらつき σ_u）≈ ([\d.]+)°", out_next)
ok(1.5 < sd < 3, f"--noise-sd なしでは、滴の平均の標準誤差に --piece-sd の既定 1.5° を足した値（{sd}°）を使う")

手順 17 では、手順 16 で求めた観測雑音（同じ条件で作った2枚の試料片の平均の差 $d$ から $|d|/\sqrt2$）を `--noise-sd` に渡して、提案された条件を1つずつ行います。ここでは形だけ、練習用の `bo_runs_demo.csv` に `--noise-sd 2.0` を渡して実行します（教科書の `bo_next.py bo_runs.csv --noise-sd 2.0` の `bo_runs.csv` を置き換えたもの）。最後は「今の推薦」の条件で**新しい試料片を2枚**作って測り、探索に使った値で性能を言わないことに注意してください。

In [ ]:
out_next2 = run("bo_next.py bo_runs_demo.csv --noise-sd 2.0")   # 教科書：bo_next.py bo_runs.csv --noise-sd 2.0（2.0 は |d|/√2 に置き換える）

In [ ]:
ok("今の推薦（事後平均が最大）：" in out_next2 and "観測雑音の標準偏差（--noise-sd）= 2.00°" in out_next2, "--noise-sd 2.0 を使って、次の候補と今の推薦が表示された")

## おわりに

- 答えのわかった合成の滴の画像で、接線法・楕円・円・θ/2 法の違い（重力で円と θ/2 法が小さく出ること）と、映り込みの画像で 90° 付近がとばされることを確かめました。
- 要因計画では、試料片を単位にした分散分析（正しい）と、滴を単位にした分散分析（疑似反復）で、同じ効果の区間と p がどれだけ違うか、交互作用があるとき主効果だけで語れないことを見ました。
- 疎水性回復では、観測した時間が τ より短いと θ∞ と τ を同時に決められないことを、ベイズ最適化では、BO の強さと失敗、勝者の呪い、検定の選び方を見ました。
- 5 節を教科書の本番（`REPS = 30`）にすると、8-7 と図8-12 の値（regret の中央値 BO 0.6°・ランダム探索 15.3°・格子 17.2°、p ≈ 0.004 と 0.06）と比べられます。
- このノートブックに入れていないもの：撮影台を組んで実物の水滴を撮ること、自分の写真の測定とその解析（手順 B〜F の実物の部分）。これらは Mac で教科書の手順のとおりに行います。